<a href="https://colab.research.google.com/github/sadiamunawar324/BrainTumor-XAI/blob/main/BT_XAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import all necessary libraries
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, LearningRateScheduler
from tensorflow.keras.regularizers import l2
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import zipfile
import shutil

In [ ]:
# ============================================================
# BRAIN MASKING UTILITIES
# Grad-CAM / SHAP heatmaps were highlighting regions outside the
# skull. Every image is masked before training AND during XAI
# visualization so models and explanations never see background.
# ============================================================

import cv2
from skimage.morphology import remove_small_objects, binary_closing, disk
from skimage.measure import label as sk_label

def create_brain_mask(img_array, threshold_factor=0.1):
    """
    Create a binary brain mask from an RGB MRI image.
    Handles both [0,1] float and [0,255] uint8 inputs.
    """
    # Normalise to [0,1] if needed
    if img_array.max() > 1.5:
        img_norm = img_array.astype(np.float32) / 255.0
    else:
        img_norm = img_array.astype(np.float32)

    gray      = np.mean(img_norm, axis=-1)
    gray_u8   = (gray * 255).astype(np.uint8)
    _, binary = cv2.threshold(gray_u8, 0, 255,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask      = binary.astype(bool)
    mask      = binary_closing(mask, disk(5))

    labeled   = sk_label(mask)
    if labeled.max() > 0:
        sizes          = np.bincount(labeled.ravel())
        sizes[0]       = 0
        largest        = sizes.argmax()
        mask           = (labeled == largest)

    mask = remove_small_objects(mask, min_size=500)
    return mask


def apply_brain_mask_to_heatmap(heatmap, mask):
    """Zero-out heatmap values outside the brain mask and re-normalise."""
    masked = heatmap.copy()
    masked[~mask] = 0.0
    if masked.max() > 0:
        masked = masked / masked.max()
    return masked


def mask_and_save_image(src_path, dst_path):
    """Load image, apply brain mask, save masked result (background = black)."""
    img = cv2.imread(src_path)
    if img is None:
        return False
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_norm = img_rgb.astype(np.float32) / 255.0

    mask = create_brain_mask(img_norm)

    # Zero out background
    img_masked = img_rgb.copy()
    img_masked[~mask] = 0

    cv2.imwrite(dst_path, cv2.cvtColor(img_masked, cv2.COLOR_RGB2BGR))
    return True


print("Brain masking utilities loaded.")
print("  create_brain_mask()      – binary brain mask from MRI")
print("  apply_brain_mask_to_heatmap() – suppresses XAI scores outside brain")
print("  mask_and_save_image()    – masks an image file and saves it")

In [ ]:
# Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)


# Define paths
ZIP_PATH = "BrainTumour.zip"
EXTRACT_PATH = "BrainTumour"
ORIGINAL_DATA_DIR = os.path.join(EXTRACT_PATH, "brain-tumor-mri-dataset")
SPLIT_DATA_DIR = os.path.join(EXTRACT_PATH, "split_dataset")


# Extract dataset
print("Extracting dataset...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset extracted successfully!")


# Find all image files
def find_image_files(base_path):
    image_files = []
    labels = []
    class_names = []

    for class_name in os.listdir(base_path):
        class_path = os.path.join(base_path, class_name)
        if os.path.isdir(class_path):
            class_names.append(class_name)
            for img_file in os.listdir(class_path):
                if img_file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                    image_files.append(os.path.join(class_path, img_file))
                    labels.append(class_name)

    return image_files, labels, sorted(class_names)



# Search for images
all_image_files, all_labels, class_names = find_image_files(ORIGINAL_DATA_DIR)
print(f"Found {len(all_image_files)} total images")
print(f"Classes: {class_names}")


# Split data into 70% training and 30% testing
X_train, X_test, y_train, y_test = train_test_split(
    all_image_files, all_labels, test_size=0.3, random_state=42, stratify=all_labels
)

print(f"Training set: {len(X_train)} images (70%)")
print(f"Testing set: {len(X_test)} images (30%)")



# Create directories for the new split
train_dir = os.path.join(SPLIT_DATA_DIR, 'train')
test_dir = os.path.join(SPLIT_DATA_DIR, 'test')

if os.path.exists(SPLIT_DATA_DIR):
    shutil.rmtree(SPLIT_DATA_DIR)

for cls in class_names:
    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(test_dir, cls), exist_ok=True)


# Copy files to new directories
def copy_files_to_directories(file_paths, labels, target_dir):
    for file_path, label in zip(file_paths, labels):
        filename = os.path.basename(file_path)
        target_path = os.path.join(target_dir, label, filename)
        shutil.copy2(file_path, target_path)

print("Copying training files...")
copy_files_to_directories(X_train, y_train, train_dir)
print("Copying testing files...")
copy_files_to_directories(X_test, y_test, test_dir)
print("Files copied successfully!")

# ============================================================
# CREATE BRAIN-MASKED DATASET FOR TRAINING / VALIDATION / TESTING
# Every image in the train and test splits is masked
# so that all pixels outside the skull are set to black (0).
# ============================================================

MASKED_SPLIT_DATA_DIR = os.path.join(EXTRACT_PATH, "split_dataset_masked")
masked_train_dir = os.path.join(MASKED_SPLIT_DATA_DIR, 'train')
masked_test_dir  = os.path.join(MASKED_SPLIT_DATA_DIR, 'test')

if os.path.exists(MASKED_SPLIT_DATA_DIR):
    shutil.rmtree(MASKED_SPLIT_DATA_DIR)

for cls in class_names:
    os.makedirs(os.path.join(masked_train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(masked_test_dir, cls), exist_ok=True)

print("Creating brain-masked training set...")
for cls in class_names:
    src_dir = os.path.join(train_dir, cls)
    dst_dir = os.path.join(masked_train_dir, cls)
    files = [f for f in os.listdir(src_dir)
             if f.lower().endswith(('.png','.jpg','.jpeg','.bmp','.tiff'))]
    for f in files:
        mask_and_save_image(os.path.join(src_dir, f),
                            os.path.join(dst_dir, f))
    print(f"  {cls}: {len(files)} images masked")

print("\nCreating brain-masked test set...")
for cls in class_names:
    src_dir = os.path.join(test_dir, cls)
    dst_dir = os.path.join(masked_test_dir, cls)
    files = [f for f in os.listdir(src_dir)
             if f.lower().endswith(('.png','.jpg','.jpeg','.bmp','.tiff'))]
    for f in files:
        mask_and_save_image(os.path.join(src_dir, f),
                            os.path.join(dst_dir, f))
    print(f"  {cls}: {len(files)} images masked")

print(f"\nMasked dataset saved to: {MASKED_SPLIT_DATA_DIR}")
print("All background pixels outside the brain are set to black.")

# Data preprocessing
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.3,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    validation_split=0.2
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    masked_train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

validation_generator = train_datagen.flow_from_directory(
    masked_train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=True,
    seed=42
)

test_generator = val_test_datagen.flow_from_directory(
    masked_test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)



# Calculate class weights
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_dict = dict(enumerate(class_weights))
print("Class weights:", class_weights_dict)

In [ ]:
# Build ResNet50 model
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

base_model.trainable = False
for layer in base_model.layers[-100:]:
    layer.trainable = True



# Custom head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(512, activation='relu', kernel_regularizer=l2(0.001))(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

predictions = Dense(len(class_names), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)




# Learning rate scheduler
def lr_scheduler(epoch, lr):
    if epoch < 10:
        return lr
    elif epoch < 20:
        return lr * 0.5
    else:
        return lr * 0.1



# Compile model
model.compile(
    optimizer=Adam(learning_rate=0.0001, clipvalue=0.5),
    loss='categorical_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.AUC(name='auc')]
)

model.summary()



# Callbacks
early_stop = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1, mode='max')
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)
checkpoint = ModelCheckpoint('best_resnet50_model.h5', monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
lr_scheduler_cb = LearningRateScheduler(lr_scheduler, verbose=1)



# Train model
print("\n--- Training Model ---")
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    epochs=45,
    callbacks=[early_stop, reduce_lr, checkpoint, lr_scheduler_cb],
    class_weight=class_weights_dict,
    verbose=1
)



# Fine-tuning phase
print("\n Fine-tuning Phase ")
for layer in base_model.layers[-150:]:
    layer.trainable = True

model.compile(
    optimizer=Adam(learning_rate=0.00001, clipvalue=0.5),
    loss='categorical_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.AUC(name='auc')]
)

history_fine = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    epochs=15,
    callbacks=[early_stop, reduce_lr, checkpoint],
    class_weight=class_weights_dict,
    verbose=1
)



# Combine training histories
combined_history = {}
for key in history.history.keys():
    combined_history[key] = history.history[key] + history_fine.history[key]



# Evaluation
print("\n Evaluation on Test Set ")
test_loss, test_accuracy, test_precision, test_recall, test_auc = model.evaluate(test_generator)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test AUC: {test_auc:.4f}")



# Generate predictions
test_generator.reset()
predictions = model.predict(test_generator, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes


# Classification report
print("\n Classification Report ")
print(classification_report(true_classes, predicted_classes, target_names=class_names, digits=4))


# AUC scores
y_true_binarized = label_binarize(true_classes, classes=range(len(class_names)))
print("\n AUC Scores (One-vs-Rest) ")
auc_scores = {}
for i, class_name in enumerate(class_names):
    class_auc = roc_auc_score(y_true_binarized[:, i], predictions[:, i])
    auc_scores[class_name] = class_auc
    print(f"{class_name}: {class_auc:.4f}")

macro_auc = roc_auc_score(y_true_binarized, predictions, multi_class='ovr')
print(f"Macro-average AUC: {macro_auc:.4f}")


# F1-score calculation
report = classification_report(true_classes, predicted_classes, target_names=class_names, output_dict=True)
print("\n F1-Scores ")
for class_name in class_names:
    print(f"{class_name}: {report[class_name]['f1-score']:.4f}")



from sklearn.metrics import mean_squared_error, mean_absolute_error, cohen_kappa_score, fbeta_score

# Dataset distribution
def plot_dataset_distribution(train_gen, test_gen, class_names):
    # Use np.bincount to count occurrences in NumPy arrays
    train_counts = np.bincount(train_gen.labels)
    test_counts = np.bincount(test_gen.classes)

    plt.figure(figsize=(8, 6))
    x = np.arange(len(class_names))
    # Ensure counts have the same length as class_names
    plt.bar(x - 0.2, train_counts[:len(class_names)], width=0.4, label='Train')
    plt.bar(x + 0.2, test_counts[:len(class_names)], width=0.4, label='Test')
    plt.xticks(x, class_names)
    plt.title("Dataset Distribution")
    plt.ylabel("Number of Images")
    plt.legend()
    plt.show()

plot_dataset_distribution(train_generator, test_generator, class_names)


# Accuracy and Loss During Training
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(combined_history['accuracy'], label='Train Accuracy')
plt.plot(combined_history['val_accuracy'], label='Val Accuracy')
plt.title("Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(combined_history['loss'], label='Train Loss')
plt.plot(combined_history['val_loss'], label='Val Loss')
plt.title("Model Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()


# Classification Report (visual)
report_df = pd.DataFrame(report).transpose()
plt.figure(figsize=(8, 4))
sns.heatmap(report_df.iloc[:-1, :-1], annot=True, cmap="Blues", fmt=".2f")
plt.title("Classification Report (Heatmap)")
plt.show()


# Confusion Matrix
cm = confusion_matrix(true_classes, predicted_classes)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


# Error Metrics
mse = mean_squared_error(true_classes, predicted_classes)
rmse = np.sqrt(mse)
mae = mean_absolute_error(true_classes, predicted_classes)

print(f"\nError Metrics:\nMSE: {mse:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")

plt.figure(figsize=(6, 4))
metrics = {'MSE': mse, 'RMSE': rmse, 'MAE': mae}
sns.barplot(x=list(metrics.keys()), y=list(metrics.values()), palette="coolwarm")
plt.title("Error Metrics")
plt.ylabel("Value")
plt.show()


# Cohen's Kappa & F2 Score (held-out test set, single value)
# Per-epoch predictions are not stored, so this is reported once
# on the final test-set predictions rather than as a fake
# "epoch-wise" trend (a flat line repeating the final value would
# be misleading, not a real training curve).
y_true_bin = label_binarize(true_classes, classes=range(len(class_names)))
f2_test = fbeta_score(true_classes, predicted_classes, beta=2, average='macro')
kappa_test = cohen_kappa_score(true_classes, predicted_classes)

print(f"ResNet50 test-set F2-Score (macro): {f2_test:.4f}")
print(f"ResNet50 test-set Cohen's Kappa:    {kappa_test:.4f}")

In [ ]:
# DenseNet121 Model (using same preprocessed data)
from tensorflow.keras.applications import DenseNet121


# Build DenseNet121 base
base_model_dn = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)


# Freeze most layers, unfreeze last 100 for fine-tuning
base_model_dn.trainable = False
for layer in base_model_dn.layers[-100:]:
    layer.trainable = True


# Custom classifier head
x = base_model_dn.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(512, activation='relu', kernel_regularizer=l2(0.001))(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

predictions_dn = Dense(len(class_names), activation='softmax')(x)

model_dn = Model(inputs=base_model_dn.input, outputs=predictions_dn)


# Compile DenseNet121
model_dn.compile(
    optimizer=Adam(learning_rate=0.0001, clipvalue=0.5),
    loss='categorical_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.AUC(name='auc')]
)


# Callbacks (reuse same ones but with new checkpoint file)
checkpoint_dn = ModelCheckpoint('best_densenet121_model.h5', monitor='val_accuracy',
                                 save_best_only=True, mode='max', verbose=1)


# Train DenseNet121
history_dn = model_dn.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    epochs=45,
    callbacks=[early_stop, reduce_lr, checkpoint_dn, lr_scheduler_cb],
    class_weight=class_weights_dict,
    verbose=1
)



# Fine-tuning (unfreeze last 150 layers)
for layer in base_model_dn.layers[-150:]:
    layer.trainable = True

model_dn.compile(
    optimizer=Adam(learning_rate=0.00001, clipvalue=0.5),
    loss='categorical_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(),
             tf.keras.metrics.Recall(),
             tf.keras.metrics.AUC()]
)
history_fine_dn = model_dn.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    epochs=15,
    callbacks=[early_stop, reduce_lr, checkpoint_dn],
    class_weight=class_weights_dict,
    verbose=1
)


# Merge training histories
combined_history_dn = {}

for key in history_dn.history.keys():
    # Get corresponding fine-tuning key (sometimes has _1 suffix)
    fine_key = key
    if fine_key not in history_fine_dn.history and (key + "_1") in history_fine_dn.history:
        fine_key = key + "_1"

    if fine_key in history_fine_dn.history:
        combined_history_dn[key] = history_dn.history[key] + history_fine_dn.history[fine_key]
    else:
        combined_history_dn[key] = history_dn.history[key]


# Evaluation on Test Set
test_loss_dn, test_acc_dn, test_prec_dn, test_rec_dn, test_auc_dn = model_dn.evaluate(test_generator)
print(f"\nDenseNet121 Results -> Loss: {test_loss_dn:.4f}, Acc: {test_acc_dn:.4f}, "
      f"Precision: {test_prec_dn:.4f}, Recall: {test_rec_dn:.4f}, AUC: {test_auc_dn:.4f}")


# Predictions
test_generator.reset()
predictions_dn = model_dn.predict(test_generator, verbose=1)
predicted_classes_dn = np.argmax(predictions_dn, axis=1)
true_classes_dn = test_generator.classes
print("\nDenseNet121 Classification Report:")
print(classification_report(true_classes_dn, predicted_classes_dn, target_names=class_names, digits=4))


# ROC-AUC
y_true_binarized_dn = label_binarize(true_classes_dn, classes=range(len(class_names)))
macro_auc_dn = roc_auc_score(y_true_binarized_dn, predictions_dn, multi_class='ovr')
print(f"DenseNet121 Macro-average AUC: {macro_auc_dn:.4f}")



from sklearn.metrics import mean_squared_error, mean_absolute_error, cohen_kappa_score, fbeta_score


# 1. Accuracy and Loss During Training
plt.figure(figsize=(12, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(combined_history_dn['accuracy'], label='Train Accuracy')
plt.plot(combined_history_dn['val_accuracy'], label='Val Accuracy')
plt.title("DenseNet121 Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(combined_history_dn['loss'], label='Train Loss')
plt.plot(combined_history_dn['val_loss'], label='Val Loss')
plt.title("DenseNet121 Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.show()


# 2. Classification Report (Heatmap)
report_dn = classification_report(true_classes_dn, predicted_classes_dn, target_names=class_names, output_dict=True)
report_df_dn = pd.DataFrame(report_dn).transpose()

plt.figure(figsize=(8, 4))
sns.heatmap(report_df_dn.iloc[:-1, :-1], annot=True, cmap="Blues", fmt=".2f")
plt.title("DenseNet121 Classification Report (Heatmap)")
plt.show()


# 3. Confusion Matrix
cm_dn = confusion_matrix(true_classes_dn, predicted_classes_dn)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_dn, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.title("DenseNet121 Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()



# 4. Error Metrics
mse_dn = mean_squared_error(true_classes_dn, predicted_classes_dn)
rmse_dn = np.sqrt(mse_dn)
mae_dn = mean_absolute_error(true_classes_dn, predicted_classes_dn)

print(f"\nDenseNet121 Error Metrics:\nMSE: {mse_dn:.4f} | RMSE: {rmse_dn:.4f} | MAE: {mae_dn:.4f}")

plt.figure(figsize=(6, 4))
metrics_dn = {'MSE': mse_dn, 'RMSE': rmse_dn, 'MAE': mae_dn}
sns.barplot(x=list(metrics_dn.keys()), y=list(metrics_dn.values()), palette="coolwarm")
plt.title("DenseNet121 Error Metrics")
plt.ylabel("Value")
plt.show()


# 5. Cohen's Kappa & F2 Score (held-out test set, single value)
# Reported once on the final test-set predictions rather than as
# a fake "epoch-wise" trend, since per-epoch predictions were
# never stored during training.
f2_test_dn = fbeta_score(true_classes_dn, predicted_classes_dn, beta=2, average='macro')
kappa_test_dn = cohen_kappa_score(true_classes_dn, predicted_classes_dn)

print(f"DenseNet121 test-set F2-Score (macro): {f2_test_dn:.4f}")
print(f"DenseNet121 test-set Cohen's Kappa:    {kappa_test_dn:.4f}")

In [ ]:
# CNN-XGBoost Hybrid Model for Brain Tumor Classification
# Custom Residual CNN + XGBoost Ensemble

import xgboost as xgb
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, BatchNormalization,
                                     Dropout, GlobalAveragePooling2D, Dense,
                                     Input, Activation, Add)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
import numpy as np
import pandas as pd
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                            cohen_kappa_score, fbeta_score, mean_squared_error, mean_absolute_error)
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from imblearn.over_sampling import ADASYN
from sklearn.feature_selection import SelectKBest, mutual_info_classif
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

print("\n" + "="*70)
print("CNN-XGBOOST HYBRID MODEL")
print("="*70)


# Build Residual CNN Architecture
print("\n[1/5] Building CNN with residual connections...")

def residual_block(x, filters, reg=0.001):
    shortcut = x
    x = Conv2D(filters, (3, 3), padding='same', kernel_regularizer=l2(reg))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, (3, 3), padding='same', kernel_regularizer=l2(reg))(x)
    x = BatchNormalization()(x)

    if shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, (1, 1), kernel_regularizer=l2(reg))(shortcut)
        shortcut = BatchNormalization()(shortcut)

    x = Add()([x, shortcut])
    return Activation('relu')(x)

def build_cnn(input_shape=(224, 224, 3), num_classes=4):
    inputs = Input(shape=input_shape)

    x = Conv2D(64, (7, 7), strides=2, padding='same', kernel_regularizer=l2(0.001))(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)

    x = residual_block(x, 64)
    x = residual_block(x, 64)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.3)(x)

    x = residual_block(x, 128)
    x = residual_block(x, 128)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.4)(x)

    x = residual_block(x, 256)
    x = residual_block(x, 256)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.5)(x)

    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu', kernel_regularizer=l2(0.002))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.6)(x)

    features = Dense(512, activation='relu', name='features', kernel_regularizer=l2(0.002))(x)
    features = BatchNormalization()(features)
    features = Dropout(0.5)(features)

    outputs = Dense(num_classes, activation='softmax')(features)
    return Model(inputs=inputs, outputs=outputs)

cnn_model = build_cnn(num_classes=len(class_names))
cnn_model.compile(optimizer=Adam(0.0005), loss='categorical_crossentropy', metrics=['accuracy'])
print(f"CNN created: {cnn_model.count_params():,} parameters")


# Train CNN
print("\n[2/5] Training CNN...")

callbacks = [
    EarlyStopping('val_accuracy', patience=20, restore_best_weights=True, mode='max'),
    ReduceLROnPlateau('val_loss', factor=0.3, patience=7, min_lr=1e-8),
    ModelCheckpoint('best_cnn_xgb.h5', 'val_accuracy', save_best_only=True, mode='max')
]

history = cnn_model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    epochs=60,
    callbacks=callbacks,
    class_weight=class_weights_dict,
    verbose=1
)


# Extract Features with Test-Time Augmentation
print("\n[3/5] Extracting features with TTA...")

feature_extractor = Model(cnn_model.input, cnn_model.get_layer('features').output)

def extract_features(generator, extractor, n_tta=3):
    features, labels = [], []
    generator.reset()
    steps = int(np.ceil(generator.samples / generator.batch_size))

    for step in range(steps):
        x_batch, y_batch = generator[step]
        tta_feats = [extractor.predict(x_batch, verbose=0) for _ in range(n_tta)]
        features.append(np.mean(tta_feats, axis=0))
        labels.append(y_batch)

    features = np.vstack(features)[:generator.samples]
    labels = np.vstack(labels)[:generator.samples]
    return features, np.argmax(labels, axis=1)

X_train, y_train = extract_features(train_generator, feature_extractor, n_tta=2)
X_val, y_val = extract_features(validation_generator, feature_extractor, n_tta=2)
X_test, y_test = extract_features(test_generator, feature_extractor, n_tta=5)
print(f"Features extracted: {X_train.shape}")


# Feature Engineering and Preprocessing
print("\n[4/5] Feature engineering...")

# Polynomial and interaction features
X_train_eng = np.column_stack([
    X_train, np.square(X_train), np.sqrt(np.abs(X_train) + 1e-8),
    np.log1p(np.abs(X_train)), X_train * np.roll(X_train, 1, axis=1)
])
X_val_eng = np.column_stack([
    X_val, np.square(X_val), np.sqrt(np.abs(X_val) + 1e-8),
    np.log1p(np.abs(X_val)), X_val * np.roll(X_val, 1, axis=1)
])
X_test_eng = np.column_stack([
    X_test, np.square(X_test), np.sqrt(np.abs(X_test) + 1e-8),
    np.log1p(np.abs(X_test)), X_test * np.roll(X_test, 1, axis=1)
])

# Scaling and feature selection
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_eng)
X_val_scaled = scaler.transform(X_val_eng)
X_test_scaled = scaler.transform(X_test_eng)

selector = SelectKBest(mutual_info_classif, k=min(400, X_train_scaled.shape[1]))
X_train_sel = selector.fit_transform(X_train_scaled, y_train)
X_val_sel = selector.transform(X_val_scaled)
X_test_sel = selector.transform(X_test_scaled)

# ADASYN balancing
adasyn = ADASYN(random_state=42, n_neighbors=5)
X_train_bal, y_train_bal = adasyn.fit_resample(X_train_sel, y_train)
print(f"ADASYN: {len(y_train)} → {len(y_train_bal)} samples")

# PCA
pca = PCA(n_components=0.99, random_state=42)
X_train_pca = pca.fit_transform(X_train_bal)
X_val_pca = pca.transform(X_val_sel)
X_test_pca = pca.transform(X_test_sel)
print(f"PCA: {X_train_bal.shape[1]} → {X_train_pca.shape[1]} features")


# Train XGBoost Ensemble
print("\n[5/5] Training XGBoost ensemble...")

configs = [
    {'name': 'Deep', 'max_depth': 15, 'lr': 0.015, 'n_est': 3000, 'gamma': 0.4, 'alpha': 1.0, 'lambda': 4.0},
    {'name': 'Wide', 'max_depth': 8, 'lr': 0.02, 'n_est': 2500, 'gamma': 0.2, 'alpha': 0.5, 'lambda': 2.0},
    {'name': 'Balanced', 'max_depth': 11, 'lr': 0.018, 'n_est': 2800, 'gamma': 0.3, 'alpha': 0.7, 'lambda': 3.0}
]

models = []
for cfg in configs:
    print(f"  Training XGB-{cfg['name']}...")
    model = xgb.XGBClassifier(
        objective='multi:softprob', num_class=len(class_names),
        max_depth=cfg['max_depth'], learning_rate=cfg['lr'], n_estimators=cfg['n_est'],
        gamma=cfg['gamma'], reg_alpha=cfg['alpha'], reg_lambda=cfg['lambda'],
        subsample=0.8, colsample_bytree=0.8, tree_method='hist',
        early_stopping_rounds=150, random_state=42, n_jobs=-1
    )
    model.fit(X_train_pca, y_train_bal, eval_set=[(X_val_pca, y_val)], verbose=100)
    models.append(model)

# Ensemble prediction
predictions = [m.predict_proba(X_test_pca) for m in models]
y_pred_proba = np.mean(predictions, axis=0)
y_pred = np.argmax(y_pred_proba, axis=1)

accuracy = accuracy_score(y_test, y_pred)
print(f"\n{'='*70}")
print(f"FINAL ACCURACY: {accuracy*100:.2f}% {'✓' if accuracy >= 0.85 else ''}")
print(f"{'='*70}\n")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))


# VISUALIZATIONS

# 1. Accuracy and Loss During Training
plt.figure(figsize=(12, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title("CNN-XGBoost Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title("CNN-XGBoost Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.show()


# 2. Classification Report (Heatmap)
report = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)
report_df = pd.DataFrame(report).transpose()

plt.figure(figsize=(8, 4))
sns.heatmap(report_df.iloc[:-1, :-1], annot=True, cmap="Blues", fmt=".2f")
plt.title("CNN-XGBoost Classification Report (Heatmap)")
plt.show()


# 3. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.title("CNN-XGBoost Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


# 4. Error Metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print(f"\nCNN-XGBoost Error Metrics:\nMSE: {mse:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")

plt.figure(figsize=(6, 4))
metrics = {'MSE': mse, 'RMSE': rmse, 'MAE': mae}
sns.barplot(x=list(metrics.keys()), y=list(metrics.values()), palette="coolwarm")
plt.title("CNN-XGBoost Error Metrics")
plt.ylabel("Value")
plt.show()


# 5. Cohen's Kappa & F2 Score (held-out test set, single value)
# Reported once on the final test-set predictions rather than as
# a fake "epoch-wise" trend, since per-epoch predictions were
# never stored during training.
f2_final = fbeta_score(y_test, y_pred, beta=2, average='macro')
kappa_final = cohen_kappa_score(y_test, y_pred)

print(f"CNN-XGBoost test-set F2-Score (macro): {f2_final:.4f}")
print(f"CNN-XGBoost test-set Cohen's Kappa:    {kappa_final:.4f}")


# Save models
cnn_model.save('cnn_xgb_hybrid.h5')
for i, m in enumerate(models):
    m.save_model(f'xgb_model_{i+1}.json')
with open('preprocessors.pkl', 'wb') as f:
    pickle.dump({'scaler': scaler, 'selector': selector, 'pca': pca}, f)

print(f"\nModels saved. Final accuracy: {accuracy*100:.2f}%")

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score
from statsmodels.stats.contingency_tables import mcnemar

In [ ]:
test_generator.reset()
y_true = test_generator.classes

In [ ]:
# Predict probabilities
probs_dn = model_dn.predict(test_generator, verbose=1)

# Convert to class labels
y_pred_dn = np.argmax(probs_dn, axis=1)

# Sanity check
assert len(y_pred_dn) == len(y_true)

acc_dn = accuracy_score(y_true, y_pred_dn)
print(f"DenseNet121 accuracy: {acc_dn:.4f}")

In [ ]:
np.save("y_true.npy", y_true)
np.save("y_pred_densenet.npy", y_pred_dn)

In [ ]:
# y_test and y_pred already available from your pipeline
assert len(y_pred) == len(y_true)

acc_xgb = accuracy_score(y_true, y_pred)
print(f"CNN-XGBoost accuracy: {acc_xgb:.4f}")

np.save("y_pred_cnn_xgb.npy", y_pred)

In [ ]:
# Load saved predictions (optional but clean)
y_true = np.load("y_true.npy")
y_pred_dn = np.load("y_pred_densenet.npy")
y_pred_xgb = np.load("y_pred_cnn_xgb.npy")

# Correct / incorrect vectors
dn_correct = (y_pred_dn == y_true)
xgb_correct = (y_pred_xgb == y_true)

# Contingency table
n11 = np.sum(dn_correct & xgb_correct)
n10 = np.sum(dn_correct & ~xgb_correct)  # DenseNet better
n01 = np.sum(~dn_correct & xgb_correct)  # XGB better
n00 = np.sum(~dn_correct & ~xgb_correct)

table = [[n11, n10],
         [n01, n00]]

print("McNemar table:", table)

In [ ]:
result = mcnemar(table, exact=False, correction=True)

print(f"Chi-square: {result.statistic:.4f}")
print(f"p-value: {result.pvalue:.6f}")

In [ ]:
# ============================================================
# McNEMAR'S TEST — All three model pairs
# Statistical significance testing across all pairwise model
# comparisons (not just DenseNet vs XGBoost).
# ============================================================

import numpy as np
from sklearn.metrics import accuracy_score
from statsmodels.stats.contingency_tables import mcnemar
from tensorflow.keras.models import load_model

# ---------- 1. Ground-truth labels ----------
test_generator.reset()
y_true = test_generator.classes

# ---------- 2. ResNet50 predictions ----------
try:
    _rn_model = load_model('best_resnet50_model.h5')
    print("ResNet50 loaded from file")
except Exception:
    _rn_model = model
    print("ResNet50 using in-memory variable 'model'")

test_generator.reset()
_probs_rn   = _rn_model.predict(test_generator, verbose=0)
y_pred_rn   = np.argmax(_probs_rn, axis=1)
assert len(y_pred_rn) == len(y_true)
print(f"ResNet50    accuracy : {accuracy_score(y_true, y_pred_rn):.4f}")

# ---------- 3. DenseNet121 predictions ----------
try:
    _dn_model = load_model('best_densenet121_model.h5')
    print("DenseNet121 loaded from file")
except Exception:
    _dn_model = model_dn
    print("DenseNet121 using in-memory variable 'model_dn'")

test_generator.reset()
_probs_dn_mc  = _dn_model.predict(test_generator, verbose=0)
y_pred_dn     = np.argmax(_probs_dn_mc, axis=1)
assert len(y_pred_dn) == len(y_true)
print(f"DenseNet121 accuracy : {accuracy_score(y_true, y_pred_dn):.4f}")

# ---------- 4. CNN-XGBoost predictions ----------
try:
    y_pred_xgb = y_pred
    assert len(y_pred_xgb) == len(y_true)
    print(f"CNN-XGBoost accuracy : {accuracy_score(y_true, y_pred_xgb):.4f}")
except (NameError, AssertionError):
    y_pred_xgb = np.load("y_pred_cnn_xgb.npy")
    print(f"CNN-XGBoost accuracy (from file): {accuracy_score(y_true, y_pred_xgb):.4f}")

# ---------- 5. Helper ----------
def _contingency_table(y_true, pred_a, pred_b):
    a_ok = (pred_a == y_true)
    b_ok = (pred_b == y_true)
    return [[int(np.sum( a_ok &  b_ok)), int(np.sum( a_ok & ~b_ok))],
            [int(np.sum(~a_ok &  b_ok)), int(np.sum(~a_ok & ~b_ok))]]

# ---------- 6. Run all three pairs ----------
_pairs = [
    ("ResNet50",    "DenseNet121",   y_pred_rn,  y_pred_dn),
    ("ResNet50",    "CNN-XGBoost",   y_pred_rn,  y_pred_xgb),
    ("DenseNet121", "CNN-XGBoost",   y_pred_dn,  y_pred_xgb),
]

print("\n" + "="*65)
print("McNEMAR'S TEST RESULTS  (Edwards continuity correction)")
print("="*65)
print(f"{'Pair':<30} {'χ²':>8}  {'p-value':>10}  {'Sig. (α=0.05)':>14}")
print("-"*65)

mcnemar_results = {}
alpha = 0.05

for name_a, name_b, pred_a, pred_b in _pairs:
    table  = _contingency_table(y_true, pred_a, pred_b)
    result = mcnemar(table, exact=False, correction=True)
    sig    = "YES ✓" if result.pvalue < alpha else "no"
    key    = f"{name_a} vs {name_b}"
    mcnemar_results[key] = {'chi2': result.statistic,
                            'pvalue': result.pvalue,
                            'table': table}
    print(f"{key:<30} {result.statistic:>8.4f}  {result.pvalue:>10.6f}  {sig:>14}")
    n = table[0][0], table[0][1], table[1][0], table[1][1]
    print(f"  Contingency: n11={n[0]}  n10={n[1]}  n01={n[2]}  n00={n[3]}")

print("="*65)

# ---------- 7. Save ----------
np.save("y_true.npy",          y_true)
np.save("y_pred_resnet.npy",   y_pred_rn)
np.save("y_pred_densenet.npy", y_pred_dn)
np.save("y_pred_cnn_xgb.npy", y_pred_xgb)
print("\nPredictions saved.")

In [ ]:
# ============================================================
# PER-CLASS McNEMAR'S TEST
# Thorough class-level statistical comparison between models,
# with particular focus on meningioma (the hardest class).
# ============================================================

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.contingency_tables import mcnemar

try:
    _ = y_true
except NameError:
    y_true     = np.load("y_true.npy")
    y_pred_rn  = np.load("y_pred_resnet.npy")
    y_pred_dn  = np.load("y_pred_densenet.npy")
    y_pred_xgb = np.load("y_pred_cnn_xgb.npy")

alpha = 0.05
_pairs_pc = [
    ("ResNet50",    "DenseNet121",   y_pred_rn,  y_pred_dn),
    ("ResNet50",    "CNN-XGBoost",   y_pred_rn,  y_pred_xgb),
    ("DenseNet121", "CNN-XGBoost",   y_pred_dn,  y_pred_xgb),
]

records = []
print("="*75)
print("PER-CLASS McNEMAR'S TEST")
print("="*75)

for class_idx, cname in enumerate(class_names):
    print(f"\n--- Class: {cname.upper()} ---")
    cmask = (y_true == class_idx)
    print(f"  N samples: {cmask.sum()}")

    for name_a, name_b, pred_a, pred_b in _pairs_pc:
        yt = y_true[cmask]; pa = pred_a[cmask]; pb = pred_b[cmask]
        a_ok = (pa == yt); b_ok = (pb == yt)
        n11 = int(np.sum( a_ok &  b_ok))
        n10 = int(np.sum( a_ok & ~b_ok))
        n01 = int(np.sum(~a_ok &  b_ok))
        n00 = int(np.sum(~a_ok & ~b_ok))
        table = [[n11, n10], [n01, n00]]

        if (n10 + n01) == 0:
            chi2, pval, sig = np.nan, np.nan, "n/a"
        else:
            exact = (n10 + n01) < 25
            res   = mcnemar(table, exact=exact, correction=True)
            chi2, pval = res.statistic, res.pvalue
            sig   = "YES ✓" if pval < alpha else "no"

        pair_label = f"{name_a} vs {name_b}"
        print(f"  {pair_label:<30}  χ²={chi2!s:>7}  p={pval!s:>9}  {sig}")
        records.append({'Class': cname, 'Pair': pair_label,
                        'chi2': chi2, 'p_value': pval,
                        'significant': (pval < alpha) if not np.isnan(pval) else False})

df_perclass = pd.DataFrame(records)

# Heatmaps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax_idx, metric in enumerate(['p_value', 'chi2']):
    pivot = df_perclass.pivot_table(index='Pair', columns='Class', values=metric, aggfunc='first')
    ordered = (['meningioma'] if 'meningioma' in pivot.columns else []) + \
              [c for c in pivot.columns if c != 'meningioma']
    pivot = pivot[ordered]

    cmap  = 'RdYlGn_r' if metric == 'p_value' else 'Blues'
    title = 'p-value (red < 0.05 = significant)' if metric == 'p_value' else 'χ² statistic'
    fmt   = '.4f' if metric == 'p_value' else '.2f'

    sns.heatmap(pivot.astype(float), annot=True, fmt=fmt, cmap=cmap,
                linewidths=1.5, ax=axes[ax_idx],
                vmin=(0 if metric == 'p_value' else None),
                vmax=(1 if metric == 'p_value' else None))
    axes[ax_idx].set_title(f'Per-class McNemar — {title}', fontweight='bold', fontsize=11)
    axes[ax_idx].set_xlabel('Tumour Class', fontweight='bold')
    axes[ax_idx].set_ylabel('Model Pair',   fontweight='bold')

    if 'meningioma' in ordered:
        col_pos = ordered.index('meningioma')
        axes[ax_idx].add_patch(
            plt.Rectangle((col_pos, 0), 1, len(pivot), fill=False, edgecolor='red', lw=3))

fig.suptitle('Per-class Statistical Comparison (Red box = meningioma)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('perclass_mcnemar.png', dpi=300, bbox_inches='tight')
plt.show()

df_perclass.to_csv('perclass_mcnemar_results.csv', index=False)
print("\nSaved: perclass_mcnemar.png  |  perclass_mcnemar_results.csv")

In [ ]:
# ============================================================
# CLASS-LEVEL PERFORMANCE COMPARISON ACROSS ALL 3 MODELS
# Meningioma misclassification is examined beyond single summary
# measures like overall recall and prediction.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter

try:
    _ = y_true
except NameError:
    y_true     = np.load("y_true.npy")
    y_pred_rn  = np.load("y_pred_resnet.npy")
    y_pred_dn  = np.load("y_pred_densenet.npy")
    y_pred_xgb = np.load("y_pred_cnn_xgb.npy")

_model_preds = {
    'ResNet50'    : y_pred_rn,
    'DenseNet121' : y_pred_dn,
    'CNN-XGBoost' : y_pred_xgb,
}
_COLORS = {
    'ResNet50'    : '#FF6B6B',
    'DenseNet121' : '#4ECDC4',
    'CNN-XGBoost' : '#95A5A6',
}

# Build per-class metrics DataFrame
rows = []
for mname, preds in _model_preds.items():
    rpt = classification_report(y_true, preds, target_names=class_names, output_dict=True, digits=4)
    for cname in class_names:
        rows.append({'Model': mname, 'Class': cname,
                     'Precision': rpt[cname]['precision'],
                     'Recall'   : rpt[cname]['recall'],
                     'F1-Score' : rpt[cname]['f1-score'],
                     'Support'  : int(rpt[cname]['support'])})

df_cl = pd.DataFrame(rows)
print("Per-class F1-Scores:")
print(df_cl.pivot_table(index='Class', columns='Model', values='F1-Score').round(4).to_string())

# Multi-panel figure
metrics_to_plot = ['Precision', 'Recall', 'F1-Score']
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

meni_idx_cl = class_names.index('meningioma') if 'meningioma' in class_names else None

for col_idx, metric in enumerate(metrics_to_plot):
    ax = fig.add_subplot(gs[col_idx, 0])
    x  = np.arange(len(class_names))
    w  = 0.25
    for i, (mname, color) in enumerate(_COLORS.items()):
        vals = df_cl[df_cl['Model'] == mname].set_index('Class').loc[class_names, metric].values
        ax.bar(x + i*w, vals, w, label=mname, color=color, alpha=0.85, edgecolor='black', linewidth=1.2)
    ax.set_xticks(x + w)
    ax.set_xticklabels(class_names, rotation=15, fontsize=10)
    ax.set_ylabel(metric, fontweight='bold')
    ax.set_title(f'Per-class {metric}', fontweight='bold', fontsize=11)
    ax.set_ylim(0, 1.12)
    ax.axhline(0.9, color='grey', linestyle='--', linewidth=1, alpha=0.5)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    if meni_idx_cl is not None:
        ax.axvspan(meni_idx_cl - 0.4, meni_idx_cl + 3*w + 0.1, alpha=0.08, color='red')

# F1-Score heatmap
ax_heat = fig.add_subplot(gs[0, 1])
pivot_f1 = df_cl.pivot_table(index='Model', columns='Class', values='F1-Score')
ordered_cols = (['meningioma'] if 'meningioma' in pivot_f1.columns else []) + \
               [c for c in class_names if c != 'meningioma']
pivot_f1 = pivot_f1[ordered_cols]
sns.heatmap(pivot_f1, annot=True, fmt='.3f', cmap='RdYlGn', linewidths=1.5, ax=ax_heat, vmin=0, vmax=1,
            annot_kws={'fontsize': 10, 'fontweight': 'bold'})
ax_heat.set_title('F1-Score Heatmap (models × classes)', fontweight='bold', fontsize=11)
ax_heat.set_xlabel(''); ax_heat.set_ylabel('')

# Meningioma binary confusion matrices
if meni_idx_cl is not None:
    sub_fig, sub_axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax_i, (mname, preds) in enumerate(_model_preds.items()):
        y_bin_t = (y_true == meni_idx_cl).astype(int)
        y_bin_p = (preds  == meni_idx_cl).astype(int)
        cm_bin  = confusion_matrix(y_bin_t, y_bin_p)
        cm_norm = cm_bin.astype(float) / cm_bin.sum(axis=1, keepdims=True)
        sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='RdYlGn', ax=sub_axes[ax_i], linewidths=1.5,
                    xticklabels=['Not Meni.', 'Meningioma'], yticklabels=['Not Meni.', 'Meningioma'], vmin=0, vmax=1)
        meni_recall = cm_norm[1, 1]
        sub_axes[ax_i].set_title(f'{mname}\nMeningioma Recall = {meni_recall:.1%}', fontweight='bold', fontsize=11)
        sub_axes[ax_i].set_xlabel('Predicted', fontweight='bold')
        sub_axes[ax_i].set_ylabel('True',      fontweight='bold')

    sub_fig.suptitle('Meningioma Binary Classification — ResNet50 → DenseNet121 → CNN-XGBoost',
                     fontsize=13, fontweight='bold')
    sub_fig.tight_layout()
    sub_fig.savefig('meningioma_binary_cm.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Saved: meningioma_binary_cm.png")

    print("\n--- Meningioma misclassification analysis ---")
    for mname, preds in _model_preds.items():
        meni_mask   = (y_true == meni_idx_cl)
        confused_as = preds[meni_mask]
        counts      = Counter(confused_as)
        print(f"\n{mname}:")
        for ci, cnt in sorted(counts.items(), key=lambda x: -x[1]):
            pct    = 100 * cnt / meni_mask.sum()
            marker = " ← CORRECT" if ci == meni_idx_cl else ""
            print(f"  Predicted as {class_names[ci]:15s}: {cnt:4d} ({pct:.1f}%){marker}")

fig.suptitle('Class-level Performance (Shaded red = meningioma)', fontsize=13, fontweight='bold', y=1.01)
plt.savefig('class_level_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
df_cl.to_csv('class_level_metrics.csv', index=False)
print("Saved: class_level_comparison.png  |  class_level_metrics.csv")

In [ ]:
# Model Comparison: ResNet50 vs DenseNet121 vs CNN-XGBoost Hybrid

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, roc_auc_score,
                             confusion_matrix)
from sklearn.preprocessing import label_binarize

print("\n" + "="*70)
print("MODEL COMPARISON ANALYSIS")
print("="*70 + "\n")

sns.set_style("whitegrid")
colors = ['#3498db', '#e74c3c', '#2ecc71']


# Collect Predictions
from tensorflow.keras.models import load_model

# Load ResNet50 model
try:
    resnet_model = load_model('best_resnet50_model.h5')
    print("Loaded ResNet50 model")
except:
    print("Warning: Could not load ResNet50 model, using current trained model")
    # Use the original model variable if available from your first code
    resnet_model = None

# Load DenseNet121 model
try:
    densenet_model = load_model('best_densenet121_model.h5')
    print("Loaded DenseNet121 model")
except:
    print("Warning: Could not load DenseNet121 model")
    densenet_model = None

# Get predictions
# Both models must load successfully - silently substituting
# zero-filled predictions would produce a bogus 0% accuracy /
# undefined Kappa row instead of a clear failure.
if resnet_model is None:
    raise RuntimeError("ResNet50 model failed to load - cannot compute metrics.")
if densenet_model is None:
    raise RuntimeError("DenseNet121 model failed to load - cannot compute metrics.")

test_generator.reset()
resnet_pred_proba = resnet_model.predict(test_generator, verbose=0)
resnet_pred = np.argmax(resnet_pred_proba, axis=1)
resnet_true = test_generator.classes

test_generator.reset()
densenet_pred_proba = densenet_model.predict(test_generator, verbose=0)
densenet_pred = np.argmax(densenet_pred_proba, axis=1)
densenet_true = test_generator.classes

# CNN-XGBoost predictions (from previous step)
cnn_xgb_pred = y_pred
cnn_xgb_pred_proba = y_pred_proba
cnn_xgb_true = y_test

print("All predictions collected\n")


# Calculate Metrics
def compute_metrics(y_true, y_pred, y_pred_proba, name):
    y_true_bin = label_binarize(y_true, classes=range(len(class_names)))
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='macro'),
        'Recall': recall_score(y_true, y_pred, average='macro'),
        'F1-Score': f1_score(y_true, y_pred, average='macro'),
        'AUC': roc_auc_score(y_true_bin, y_pred_proba, multi_class='ovr'),
        'Kappa': cohen_kappa_score(y_true, y_pred)
    }

metrics = [
    compute_metrics(resnet_true, resnet_pred, resnet_pred_proba, 'ResNet50'),
    compute_metrics(densenet_true, densenet_pred, densenet_pred_proba, 'DenseNet121'),
    compute_metrics(cnn_xgb_true, cnn_xgb_pred, cnn_xgb_pred_proba, 'CNN-XGBoost')
]

df = pd.DataFrame(metrics)

# Print summary
for _, row in df.iterrows():
    print(f"{row['Model']:15s} → Accuracy: {row['Accuracy']*100:5.2f}% | "
          f"F1: {row['F1-Score']:.4f} | AUC: {row['AUC']:.4f}")
print()


# Visualizations

# 1. Overall Metrics Comparison
fig, ax = plt.subplots(figsize=(12, 5))
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC', 'Kappa']
x = np.arange(len(metric_cols))
width = 0.25

for i, (_, row) in enumerate(df.iterrows()):
    values = [row[m] for m in metric_cols]
    ax.bar(x + i*width, values, width, label=row['Model'], color=colors[i], alpha=0.85)

ax.set_xlabel('Metrics', fontweight='bold')
ax.set_ylabel('Score', fontweight='bold')
ax.set_title('Model Performance Comparison', fontweight='bold', fontsize=13)
ax.set_xticks(x + width)
ax.set_xticklabels(metric_cols)
ax.legend()
ax.set_ylim([0, 1.1])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


# 2. Accuracy Comparison
fig, ax = plt.subplots(figsize=(10, 5))
models = df['Model'].values
accuracies = df['Accuracy'].values * 100

bars = ax.barh(models, accuracies, color=colors, alpha=0.85, edgecolor='black', linewidth=1.5)
ax.set_xlabel('Accuracy (%)', fontweight='bold')
ax.set_title('Accuracy Comparison', fontweight='bold', fontsize=13)
ax.set_xlim([0, 100])
ax.axvline(x=85, color='green', linestyle='--', linewidth=2, alpha=0.6, label='Target (85%)')
ax.grid(axis='x', alpha=0.3)
ax.legend()

for bar, acc in zip(bars, accuracies):
    ax.text(acc + 1.5, bar.get_y() + bar.get_height()/2,
            f'{acc:.2f}%', va='center', fontweight='bold')

best_idx = np.argmax(accuracies)
bars[best_idx].set_edgecolor('gold')
bars[best_idx].set_linewidth(3)

plt.tight_layout()
plt.show()


# 3. Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
cms = [
    confusion_matrix(resnet_true, resnet_pred),
    confusion_matrix(densenet_true, densenet_pred),
    confusion_matrix(cnn_xgb_true, cnn_xgb_pred)
]

for ax, cm, row in zip(axes, cms, df.iterrows()):
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='RdYlGn', ax=ax,
                xticklabels=class_names, yticklabels=class_names, vmin=0, vmax=1)
    ax.set_title(f"{row[1]['Model']}\nAcc: {row[1]['Accuracy']*100:.2f}%", fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.show()


# 4. Radar Chart
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))
angles = np.linspace(0, 2*np.pi, len(metric_cols), endpoint=False).tolist()
angles += angles[:1]

for i, (_, row) in enumerate(df.iterrows()):
    values = [row[m] for m in metric_cols] + [row[metric_cols[0]]]
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Model'], color=colors[i])
    ax.fill(angles, values, alpha=0.15, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_cols)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.grid(True, alpha=0.7)
ax.set_title('Multi-Metric Radar Chart', fontweight='bold', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()


# 5. Summary Table
fig, ax = plt.subplots(figsize=(11, 3))
ax.axis('off')

table_data = [[row['Model'], f"{row['Accuracy']*100:.2f}%",
               f"{row['Precision']:.3f}", f"{row['Recall']:.3f}",
               f"{row['F1-Score']:.3f}", f"{row['AUC']:.3f}",
               f"{row['Kappa']:.3f}"] for _, row in df.iterrows()]

table = ax.table(cellText=table_data,
                colLabels=['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'AUC', 'Kappa'],
                cellLoc='center', loc='center', colColours=['#e0e0e0']*7)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)

# Highlight best values
for i, col in enumerate(metric_cols):
    best_idx = df[col].idxmax()
    table[(best_idx+1, i+1)].set_facecolor('#90EE90')
    table[(best_idx+1, i+1)].set_text_props(weight='bold')

for i in range(7):
    table[(0, i)].set_facecolor('#4CAF50')
    table[(0, i)].set_text_props(weight='bold', color='white')

plt.title('Performance Summary Table', fontweight='bold', fontsize=13, pad=15)
plt.tight_layout()
plt.show()


# Final Summary
print("="*70)
print("SUMMARY")
print("="*70)

best_model = df.loc[df['Accuracy'].idxmax(), 'Model']
best_acc = df['Accuracy'].max() * 100

print(f"\nBest Model: {best_model} ({best_acc:.2f}% accuracy)")
print(f"\nRankings by Accuracy:")
for i, (_, row) in enumerate(df.sort_values('Accuracy', ascending=False).iterrows(), 1):
    print(f"  {i}. {row['Model']:15s} - {row['Accuracy']*100:.2f}%")

# Removed the lines causing NameError

df.to_csv('model_comparison.csv', index=False)
print(f"\nResults saved to 'model_comparison.csv'")
print("="*70)

In [ ]:
import shap
from tensorflow.keras.models import load_model
from matplotlib.gridspec import GridSpec

print("="*80)
print("ENHANCED SHAP ANALYSIS FOR BRAIN TUMOR CLASSIFICATION MODELS")
print("="*80 + "\n")

# 1. Load Models
print("[Loading models")

models_info = {}

try:
    models_info['ResNet50'] = {
        'model': load_model('best_resnet50_model.h5'),
        'loaded': True
    }
    print("Loaded ResNet50")
except:
    print("Could not load ResNet50, using in-memory 'model' variable")
    try:
        models_info['ResNet50'] = {'model': model, 'loaded': True}
    except:
        models_info['ResNet50'] = {'loaded': False}

try:
    models_info['DenseNet121'] = {
        'model': load_model('best_densenet121_model.h5'),
        'loaded': True
    }
    print("Loaded DenseNet121")
except:
    print("Could not load DenseNet121, using in-memory 'model_dn' variable")
    try:
        models_info['DenseNet121'] = {'model': model_dn, 'loaded': True}
    except:
        models_info['DenseNet121'] = {'loaded': False}

try:
    models_info['CNN-XGBoost'] = {
        'model': load_model('cnn_xgb_hybrid.h5'),
        'loaded': True
    }
    print("Loaded CNN-XGBoost")
except:
    print("Could not load CNN-XGBoost, using in-memory 'cnn_model' variable")
    try:
        models_info['CNN-XGBoost'] = {'model': cnn_model, 'loaded': True}
    except:
        models_info['CNN-XGBoost'] = {'loaded': False}

# 2. Prepare Data
print("\nPreparing data")

# Get test data
test_generator.reset()
X_test_batch, y_test_batch = next(test_generator)
class_names = list(test_generator.class_indices.keys())

# Get background data (representative sample for SHAP)
train_generator.reset()
background = next(train_generator)[0][:20]  # 20 background samples

# Select images to explain, placing a meningioma case first so the
# multi-model comparison figure (Section 5 below) explains the
# hardest class rather than an arbitrary first image.
meni_idx_shap = class_names.index('meningioma')
_batch_true_shap = np.argmax(y_test_batch, axis=1)
_meni_in_batch_shap = np.where(_batch_true_shap == meni_idx_shap)[0]

if len(_meni_in_batch_shap) > 0:
    _lead_idx = int(_meni_in_batch_shap[0])
    print(f"Using meningioma image at batch index {_lead_idx} as the lead comparison image.")
else:
    test_generator.reset()
    _X_all_shap, _y_all_shap = [], []
    for _ in range(len(test_generator)):
        xb, yb = next(test_generator)
        _X_all_shap.append(xb); _y_all_shap.append(yb)
        if test_generator.batch_index == 0:
            break
    _X_all_shap = np.vstack(_X_all_shap)[:test_generator.samples]
    _y_all_shap = np.vstack(_y_all_shap)[:test_generator.samples]
    _true_all_shap = np.argmax(_y_all_shap, axis=1)
    _meni_full_shap = np.where(_true_all_shap == meni_idx_shap)[0]
    print(f"No meningioma image in current batch; substituting one from the full test set.")
    X_test_batch = np.vstack([X_test_batch, _X_all_shap[_meni_full_shap[:1]]])
    y_test_batch = np.vstack([y_test_batch, _y_all_shap[_meni_full_shap[:1]]])
    _lead_idx = len(X_test_batch) - 1

_other_idx = [i for i in range(len(X_test_batch)) if i != _lead_idx][:9]
_order = [_lead_idx] + _other_idx

X_explain = X_test_batch[_order]  # meningioma image first, then 9 others
y_explain = y_test_batch[_order]

print(f"Background samples: {background.shape}")
print(f"Images to explain: {X_explain.shape}")
print(f"Classes: {class_names}\n")


# 3. Enhanced SHAP Visualization Function ---

def visualize_shap_enhanced(model, model_name, X_explain, y_true, class_names,
                           background, num_images=5):
    """
    Enhanced SHAP visualization with multiple views and interpretations.
    """
    print(f"\n{'='*80}")
    print(f"SHAP ANALYSIS: {model_name}")
    print(f"{'='*80}")

    try:
        # Initialize DeepExplainer
        print(f"[{model_name}] Initializing SHAP DeepExplainer")
        explainer = shap.DeepExplainer(model, background)

        # Compute SHAP values
        print(f"[{model_name}] Computing SHAP values (this may take a few minutes)")
        shap_values = explainer.shap_values(X_explain[:num_images])

        # Get predictions
        predictions = model.predict(X_explain[:num_images], verbose=0)
        pred_classes = np.argmax(predictions, axis=1)
        true_classes = np.argmax(y_true[:num_images], axis=1)

        print(f"SHAP computation complete!\n")

        # Visualization 1: Individual Image Analysis
        print(f"[{model_name}] Generating individual image visualizations")

        for img_idx in range(min(3, num_images)):
            pred_idx = pred_classes[img_idx]
            true_idx = true_classes[img_idx]
            confidence = predictions[img_idx][pred_idx] * 100

            is_correct = pred_idx == true_idx
            status = "CORRECT" if is_correct else "INCORRECT"
            color = 'green' if is_correct else 'red'

            print(f"\n  Image {img_idx + 1}: {status}")
            print(f"    True: {class_names[true_idx]} | Predicted: {class_names[pred_idx]} ({confidence:.1f}%)")

            # Create comprehensive visualization
            fig = plt.figure(figsize=(18, 5))
            gs = GridSpec(1, 4, figure=fig, wspace=0.3)

            # Original image
            ax1 = fig.add_subplot(gs[0, 0])
            ax1.imshow(X_explain[img_idx])
            ax1.set_title(f'Original Image\nTrue: {class_names[true_idx]}',
                         fontweight='bold', fontsize=11)
            ax1.axis('off')

            # SHAP values for predicted class
            ax2 = fig.add_subplot(gs[0, 1])
            shap_img = shap_values[pred_idx][img_idx]
            im = ax2.imshow(np.sum(np.abs(shap_img), axis=-1), cmap='hot')
            ax2.set_title(f'SHAP Importance Map\nPredicted: {class_names[pred_idx]}',
                         fontweight='bold', fontsize=11)
            ax2.axis('off')
            plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)

            # Overlay: Positive contributions (red) and negative (blue)
            ax3 = fig.add_subplot(gs[0, 2])
            shap_rgb = np.sum(shap_img, axis=-1)
            # Normalize for visualization
            shap_rgb = (shap_rgb - shap_rgb.min()) / (shap_rgb.max() - shap_rgb.min() + 1e-8)
            ax3.imshow(X_explain[img_idx], alpha=0.6)
            im2 = ax3.imshow(shap_rgb, cmap='RdBu_r', alpha=0.5)
            ax3.set_title('SHAP Overlay\n(Red=Positive, Blue=Negative)',
                         fontweight='bold', fontsize=11)
            ax3.axis('off')

            # Confidence distribution for all classes
            ax4 = fig.add_subplot(gs[0, 3])
            bars = ax4.barh(class_names, predictions[img_idx] * 100,
                           color=['green' if i == pred_idx else 'gray'
                                  for i in range(len(class_names))])
            bars[pred_idx].set_color('green')
            if true_idx != pred_idx:
                bars[true_idx].set_color('orange')
            ax4.set_xlabel('Confidence (%)', fontweight='bold')
            ax4.set_title('Class Probabilities', fontweight='bold', fontsize=11)
            ax4.set_xlim([0, 100])
            ax4.grid(axis='x', alpha=0.3)

            fig.suptitle(f'{model_name} - SHAP Analysis (Image {img_idx + 1}) - {status}',
                        fontsize=14, fontweight='bold', color=color)
            plt.tight_layout()
            plt.show()


        # Visualization 2: SHAP Summary Plot
        print(f"\n[{model_name}] Generating SHAP summary plot")

        fig, axes = plt.subplots(1, len(class_names), figsize=(5*len(class_names), 6))
        if len(class_names) == 1:
            axes = [axes]

        for class_idx, class_name in enumerate(class_names):
            ax = axes[class_idx]

            # Get mean absolute SHAP values for this class
            shap_class = np.array(shap_values[class_idx])
            mean_shap = np.mean(np.abs(shap_class), axis=0)
            mean_shap_sum = np.sum(mean_shap, axis=-1)  # Sum across color channels

            im = ax.imshow(mean_shap_sum, cmap='hot')
            ax.set_title(f'{class_name}\nMean Feature Importance',
                        fontweight='bold', fontsize=11)
            ax.axis('off')
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        fig.suptitle(f'{model_name} - Average SHAP Importance by Class',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()


        # Visualization 3: SHAP Waterfall Plot (Feature Importance)
        print(f"\n[{model_name}] Generating feature importance analysis")

        # Calculate average feature importance across all explained images
        importance_by_class = {}
        for class_idx, class_name in enumerate(class_names):
            shap_class = np.array(shap_values[class_idx])
            # Sum absolute SHAP values across all images and pixels
            total_importance = np.sum(np.abs(shap_class))
            importance_by_class[class_name] = total_importance

        # Plot
        fig, ax = plt.subplots(figsize=(10, 6))
        classes = list(importance_by_class.keys())
        importances = list(importance_by_class.values())
        colors_bar = plt.cm.viridis(np.linspace(0, 1, len(classes)))

        bars = ax.bar(classes, importances, color=colors_bar, edgecolor='black', linewidth=1.5)
        ax.set_ylabel('Total SHAP Importance', fontweight='bold', fontsize=12)
        ax.set_xlabel('Class', fontweight='bold', fontsize=12)
        ax.set_title(f'{model_name} - Feature Importance by Class',
                    fontweight='bold', fontsize=14)
        ax.grid(axis='y', alpha=0.3)

        # Add value labels on bars
        for bar, val in zip(bars, importances):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{val:.0f}', ha='center', va='bottom', fontweight='bold')

        plt.tight_layout()
        plt.show()


        # Visualization 4: SHAP Image Plot (Standard)
        print(f"\n[{model_name}] Generating standard SHAP image plot")

        # Use the predicted class for each image
        shap.image_plot(
            [shap_values[i] for i in pred_classes[:num_images]],
            X_explain[:num_images],
            show=True
        )

        print(f"{model_name} SHAP analysis complete!\n")

        return shap_values, predictions

    except Exception as e:
        print(f"Error during SHAP analysis for {model_name}: {e}")
        return None, None


# 4. Run SHAP Analysis for All Models

results = {}

for model_name, info in models_info.items():
    if info['loaded']:
        shap_vals, preds = visualize_shap_enhanced(
            model=info['model'],
            model_name=model_name,
            X_explain=X_explain,
            y_true=y_explain,
            class_names=class_names,
            background=background,
            num_images=5
        )
        results[model_name] = {
            'shap_values': shap_vals,
            'predictions': preds
        }
    else:
        print(f"\nSkipping {model_name} (not loaded)")


# 5. Comparative Analysis

if len(results) > 1:
    print("\n" + "="*80)
    print("COMPARATIVE SHAP ANALYSIS ACROSS MODELS")
    print("="*80 + "\n")

    # Compare predictions for the same images
    img_idx = 0  # Meningioma image (placed first in X_explain above)

    fig, axes = plt.subplots(1, len(results) + 1, figsize=(5*(len(results)+1), 5))

    # Original image
    axes[0].imshow(X_explain[img_idx])
    true_class = class_names[np.argmax(y_explain[img_idx])]
    axes[0].set_title(f'Original Image\nTrue: {true_class}', fontweight='bold', fontsize=12)
    axes[0].axis('off')

    # SHAP importance for each model
    for idx, (model_name, result) in enumerate(results.items()):
        if result['shap_values'] is not None:
            pred_idx = np.argmax(result['predictions'][img_idx])
            shap_img = result['shap_values'][pred_idx][img_idx]
            importance_map = np.sum(np.abs(shap_img), axis=-1)

            im = axes[idx + 1].imshow(importance_map, cmap='hot')
            axes[idx + 1].set_title(f'{model_name}\nPred: {class_names[pred_idx]}',
                                   fontweight='bold', fontsize=12)
            axes[idx + 1].axis('off')
            plt.colorbar(im, ax=axes[idx + 1], fraction=0.046, pad=0.04)

    fig.suptitle('Model Comparison: SHAP Importance Maps (Same Image)',
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


# 6. Summary Report

print("\n" + "="*80)
print("SHAP ANALYSIS SUMMARY")
print("="*80 + "\n")

for model_name, info in models_info.items():
    if info['loaded']:
        print(f"{model_name}: Analysis complete")
    else:
        print(f"✗ {model_name}: Model not available")

print(f"\nAnalyzed {len(X_explain)} test images")
print(f"Used {len(background)} background samples")
print(f"Classes: {', '.join(class_names)}")
print("\n" + "="*80)
print("SHAP ANALYSIS COMPLETE!")
print("="*80)

In [ ]:
# SHAP - MENINGIOMA-SPECIFIC ANALYSIS
import gc
from tensorflow.keras.models import load_model
import shap
import cv2
import warnings
warnings.filterwarnings('ignore')

# Free any models/graphs left over from previous runs of this cell
try:
    tf.keras.backend.clear_session()
except:
    pass
gc.collect()

try:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision enabled")
except:
    print("Mixed precision not available")

print("MENINGIOMA-SPECIFIC SHAP ANALYSIS\n" + "="*60)


# LOAD MODELS WITH ERROR HANDLING
print("Loading models...")

try:
    resnet_model = load_model('best_resnet50_model.h5', compile=False)
    print("ResNet50 loaded")
except:
    try:
        resnet_model = model
        print("ResNet50 from memory")
    except:
        print("ResNet50 not found - skipping")
        resnet_model = None

try:
    densenet_model = load_model('best_densenet121_model.h5', compile=False)
    print("DenseNet121 loaded")
except:
    try:
        densenet_model = model_dn
        print("DenseNet121 from memory")
    except:
        print("DenseNet121 not found - skipping")
        densenet_model = None

try:
    cnn_hybrid = load_model('cnn_xgb_hybrid.h5', compile=False)
    print("CNN-XGBoost loaded")
except:
    try:
        cnn_hybrid = cnn_model
        print("CNN-XGBoost from memory")
    except:
        print("CNN-XGBoost not found - skipping")
        cnn_hybrid = None


# STREAM THE TEST SET TO FIND THE FIRST MENINGIOMA IMAGE
# No correctness filtering - fairly (not cherry-picked) selected,
# same image explained across every model. Also collects a small
# background sample for the SHAP explainer's baseline distribution
# - never the full test set.
print("\nSearching test set for a meningioma image + collecting a small background sample...")
test_generator.reset()
class_names = list(test_generator.class_indices.keys())
meni_label_idx = class_names.index('meningioma')

img = None
true_label = None
BACKGROUND_SIZE = 20
bg_imgs = []

n_batches = len(test_generator)
for b in range(n_batches):
    xb, yb = next(test_generator)
    true_labels = np.argmax(yb, axis=1)

    if img is None:
        meni_positions = np.where(true_labels == meni_label_idx)[0]
        if len(meni_positions) > 0:
            img = xb[meni_positions[0]].copy()
            true_label = meni_label_idx

    if len(bg_imgs) < BACKGROUND_SIZE:
        take = min(len(xb), BACKGROUND_SIZE - len(bg_imgs))
        bg_imgs.append(xb[:take].copy())

    del xb, yb

    if img is not None and sum(len(a) for a in bg_imgs) >= BACKGROUND_SIZE:
        break

gc.collect()

if img is None:
    raise RuntimeError("No meningioma image found in test set - check class_names / generator.")

background = np.vstack(bg_imgs)[:BACKGROUND_SIZE]
del bg_imgs
gc.collect()

print(f"Using the first meningioma image found (batch {b}).")
print(f"Background sample for SHAP: {background.shape}")
print()


# SHAP - one function, called separately per model (mirrors the
# LIME cell's explain_cnn pattern). Each call builds its own
# explainer, produces its own standalone figure, then frees the
# explainer before the next model runs - never more than one
# DeepExplainer alive in memory at a time.
def explain_shap(model, name, img, true_label, background):
    """SHAP for a single model on a single pre-selected image,
    producing its own standalone figure."""
    if model is None:
        print(f"⊗ {name} skipped (model not loaded)")
        return

    try:
        pred_proba = model.predict(img[np.newaxis, ...], verbose=0)[0]
        pred_class = np.argmax(pred_proba)

        print(f"→ {name}: True={class_names[true_label]}, Pred={class_names[pred_class]} ({pred_proba[pred_class]:.1%})")

        explainer = shap.DeepExplainer(model, background)
        shap_values = explainer.shap_values(img[np.newaxis, ...], check_additivity=False)

        # shap_values is a list (one array per class) or a single
        # array depending on shap version/model output shape
        if isinstance(shap_values, list):
            sv = shap_values[pred_class][0]
        else:
            sv = shap_values[0][..., pred_class]

        heatmap = np.sum(np.abs(sv), axis=-1)
        heatmap = heatmap / (heatmap.max() + 1e-10)  # normalize to [0, 1]

        # Apply brain mask - suppresses attribution outside the skull
        _brain_mask = create_brain_mask(img)
        heatmap = apply_brain_mask_to_heatmap(heatmap, _brain_mask)

        # Overlay (same style as the Grad-CAM cell)
        heatmap_uint8 = np.uint8(255 * heatmap)
        heatmap_colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
        img_uint8 = np.uint8(img * 255)
        img_bgr = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR)
        superimposed = cv2.addWeighted(img_bgr, 0.6, heatmap_colored, 0.4, 0)
        superimposed_rgb = cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB)

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))

        axes[0].imshow(img)
        axes[0].set_title(f'Original\n{class_names[true_label]}', fontweight='bold')
        axes[0].axis('off')

        im = axes[1].imshow(heatmap, cmap='hot')
        axes[1].set_title('SHAP Importance', fontweight='bold')
        axes[1].axis('off')
        plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

        axes[2].imshow(superimposed_rgb)
        axes[2].set_title('SHAP Overlay', fontweight='bold')
        axes[2].axis('off')

        plt.suptitle(f'{name}: {class_names[pred_class]} ({pred_proba[pred_class]:.1%})',
                    fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

        del explainer, shap_values, sv, heatmap, heatmap_colored, superimposed, superimposed_rgb
        gc.collect()

        print(f"{name} SHAP explanation complete\n")

    except Exception as e:
        print(f"Error explaining {name}: {e}\n")


print("SHAP Models (meningioma):\n" + "-"*60)

explain_shap(resnet_model, "ResNet50", img, true_label, background)
explain_shap(densenet_model, "DenseNet121", img, true_label, background)
explain_shap(cnn_hybrid, "CNN-XGBoost", img, true_label, background)

print("="*60)
print("MENINGIOMA SHAP ANALYSIS COMPLETE")
print("="*60)

In [ ]:
from tensorflow.keras.models import load_model, Model
import cv2

# 1. Core Grad-CAM Function

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Computes Grad-CAM heatmap for any model.

    Args:
        img_array (np.array): Preprocessed image array (1, H, W, C).
        model (tf.keras.Model): The trained model.
        last_conv_layer_name (str): Name of the last convolutional layer.
        pred_index (int, optional): Target class index. If None, uses predicted class.

    Returns:
        np.array: Normalized heatmap [0, 1].
    """
    # Create a model that outputs both the last conv layer and final predictions
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    # Track gradients
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)

        if pred_index is None:
            pred_index = tf.argmax(preds[0])

        # Get the output for the target class
        class_channel = preds[:, pred_index]

    # Compute gradients of the class output w.r.t. last conv layer
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # Global average pooling of gradients (importance weights)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight the feature maps by the gradients
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Apply ReLU (only positive contributions)
    heatmap = tf.maximum(heatmap, 0)

    # Normalize to [0, 1]
    max_heatmap = tf.reduce_max(heatmap)
    if max_heatmap != 0:
        heatmap /= max_heatmap

    return heatmap.numpy()


# 2. Single Image Visualization Function

def visualize_gradcam(model, model_name, img_array, class_names, last_conv_layer_name):
    """
    Generates and visualizes Grad-CAM heatmap for a single image.

    Args:
        model: Trained model.
        model_name: Name of the model (for titles).
        img_array: Input image array (1, H, W, C).
        class_names: List of class names.
        last_conv_layer_name: Name of the last convolutional layer.
    """
    # Get prediction
    preds = model.predict(img_array, verbose=0)
    pred_index = np.argmax(preds[0])
    predicted_class = class_names[pred_index]
    confidence = preds[0][pred_index] * 100

    # Generate heatmap
    heatmap = make_gradcam_heatmap(
        img_array, model, last_conv_layer_name, pred_index
    )

    # Prepare original image
    img = img_array[0]

    # Resize heatmap to match original image size
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

    # Apply brain mask — suppresses activations outside the skull
    _brain_mask     = create_brain_mask(img)
    heatmap_resized = apply_brain_mask_to_heatmap(heatmap_resized, _brain_mask)

    # Convert heatmap to RGB using JET colormap
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

    # Prepare original image for overlay
    img_uint8 = np.uint8(img * 255)
    img_bgr = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR)

    # Superimpose heatmap on original image
    superimposed = cv2.addWeighted(img_bgr, 0.6, heatmap_colored, 0.4, 0)
    superimposed_rgb = cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB)

    # Plot results
    fig = plt.figure(figsize=(15, 5))

    # Original image
    plt.subplot(1, 3, 1)
    plt.imshow(img)
    plt.title(f'Original Image\nPredicted: {predicted_class}\nConfidence: {confidence:.2f}%',
              fontsize=11, fontweight='bold')
    plt.axis('off')

    # Heatmap only
    plt.subplot(1, 3, 2)
    plt.imshow(heatmap_resized, cmap='jet')
    plt.title('Grad-CAM Heatmap\n(Feature Importance)',
              fontsize=11, fontweight='bold')
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.axis('off')

    # Superimposed
    plt.subplot(1, 3, 3)
    plt.imshow(superimposed_rgb)
    plt.title(f'Grad-CAM Overlay\n{model_name}',
              fontsize=11, fontweight='bold')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    return heatmap


# 3. Multi-Image Grad-CAM Visualization

def visualize_multiple_gradcam(model, model_name, test_generator, class_names,
                               last_conv_layer_name, num_images=6):
    """
    Visualize Grad-CAM for multiple test images in a grid.

    Args:
        model: Trained model.
        model_name: Name of the model (for title).
        test_generator: Test data generator.
        class_names: List of class names.
        last_conv_layer_name: Name of last convolutional layer.
        num_images: Number of images to visualize.
    """
    test_generator.reset()
    X_batch, y_batch = next(test_generator)

    # Determine grid size
    cols = 3
    rows = (num_images + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
    axes = axes.flatten() if num_images > 1 else [axes]

    for idx in range(min(num_images, len(X_batch))):
        img_array = X_batch[idx][np.newaxis, ...]
        true_label = class_names[np.argmax(y_batch[idx])]

        # Get prediction
        preds = model.predict(img_array, verbose=0)
        pred_index = np.argmax(preds[0])
        pred_label = class_names[pred_index]
        confidence = preds[0][pred_index] * 100

        # Generate heatmap
        heatmap = make_gradcam_heatmap(
            img_array, model, last_conv_layer_name, pred_index
        )

        # Prepare visualization
        img = img_array[0]
        heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

        # Apply brain mask — suppresses activations outside the skull
        _brain_mask     = create_brain_mask(img)
        heatmap_resized = apply_brain_mask_to_heatmap(heatmap_resized, _brain_mask)

        heatmap_uint8 = np.uint8(255 * heatmap_resized)
        heatmap_colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

        img_uint8 = np.uint8(img * 255)
        img_bgr = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR)
        superimposed = cv2.addWeighted(img_bgr, 0.6, heatmap_colored, 0.4, 0)
        superimposed_rgb = cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB)

        # Plot
        axes[idx].imshow(superimposed_rgb)

        # Color code: green if correct, red if incorrect
        color = 'green' if pred_label == true_label else 'red'
        status = '✓' if pred_label == true_label else '✗'

        axes[idx].set_title(
            f'{status} True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)',
            fontsize=10, fontweight='bold', color=color
        )
        axes[idx].axis('off')

    # Hide unused subplots
    for idx in range(num_images, len(axes)):
        axes[idx].axis('off')

    plt.suptitle(f'Grad-CAM Visualization - {model_name}',
                 fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()


# 4. Comparative Visualization (All Models)

def compare_gradcam_all_models(models_dict, img_array, class_names):
    """
    Compare Grad-CAM visualizations across all three models side-by-side.

    Args:
        models_dict: Dictionary with model info {'name': str, 'model': Model, 'layer': str}
        img_array: Input image array (1, H, W, C).
        class_names: List of class names.
    """
    num_models = len(models_dict)
    fig, axes = plt.subplots(num_models, 3, figsize=(15, 5*num_models))

    if num_models == 1:
        axes = axes.reshape(1, -1)

    img = img_array[0]

    for row, model_info in enumerate(models_dict):
        model = model_info['model']
        model_name = model_info['name']
        last_conv_layer = model_info['layer']

        # Get prediction
        preds = model.predict(img_array, verbose=0)
        pred_index = np.argmax(preds[0])
        predicted_class = class_names[pred_index]
        confidence = preds[0][pred_index] * 100

        # Generate heatmap
        heatmap = make_gradcam_heatmap(
            img_array, model, last_conv_layer, pred_index
        )

        # Prepare visualizations
        heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

        # Apply brain mask — suppresses activations outside the skull
        _brain_mask     = create_brain_mask(img)
        heatmap_resized = apply_brain_mask_to_heatmap(heatmap_resized, _brain_mask)

        heatmap_uint8 = np.uint8(255 * heatmap_resized)
        heatmap_colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

        img_uint8 = np.uint8(img * 255)
        img_bgr = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR)
        superimposed = cv2.addWeighted(img_bgr, 0.6, heatmap_colored, 0.4, 0)
        superimposed_rgb = cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB)

        # Original image
        axes[row, 0].imshow(img)
        axes[row, 0].set_title(f'{model_name}\nOriginal Image', fontweight='bold')
        axes[row, 0].axis('off')

        # Heatmap
        axes[row, 1].imshow(heatmap_resized, cmap='jet')
        axes[row, 1].set_title(f'Heatmap\n{predicted_class} ({confidence:.1f}%)', fontweight='bold')
        axes[row, 1].axis('off')

        # Overlay
        axes[row, 2].imshow(superimposed_rgb)
        axes[row, 2].set_title(f'Grad-CAM Overlay', fontweight='bold')
        axes[row, 2].axis('off')

    plt.suptitle('Model Comparison: Grad-CAM Visualizations',
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()


# 5. Load Models and Setup

print("="*70)
print("GRAD-CAM VISUALIZATION FOR ALL MODELS")
print("="*70 + "\n")

# Load models
try:
    resnet_model = load_model('best_resnet50_model.h5')
    print("Loaded ResNet50 model")
except Exception as e:
    print(f"Could not load ResNet50: {e}")
    resnet_model = None

try:
    densenet_model = load_model('best_densenet121_model.h5')
    print("Loaded DenseNet121 model")
except Exception as e:
    print(f"Could not load DenseNet121: {e}")
    densenet_model = None

try:
    cnn_xgb_model = load_model('cnn_xgb_hybrid.h5')
    print("Loaded CNN-XGBoost model")
except Exception as e:
    print(f"Could not load CNN-XGBoost: {e}")
    cnn_xgb_model = None

# Get test data
test_generator.reset()
X_test_batch, y_test_batch = next(test_generator)
class_names = list(test_generator.class_indices.keys())

# Define last convolutional layers for each model
RESNET_LAST_CONV_LAYER = "conv5_block3_out"
DENSENET_LAST_CONV_LAYER = "relu"  # or "conv5_block16_concat"
CNN_XGB_LAST_CONV_LAYER = "activation_8"  # Adjust based on your architecture

print(f"\nClass names: {class_names}")
print(f"Test batch shape: {X_test_batch.shape}\n")


# 6. Individual Model Visualizations

# Select an image to explain
image_to_explain_index = 0
image_array = X_test_batch[image_to_explain_index][np.newaxis, ...]
true_label = class_names[np.argmax(y_test_batch[image_to_explain_index])]

print(f"Analyzing image {image_to_explain_index} (True label: {true_label})")
print("="*70 + "\n")


# ResNet50 Grad-CAM
if resnet_model is not None:
    print("ResNet50 Grad-CAM")
    visualize_gradcam(
        resnet_model,
        "ResNet50",
        image_array,
        class_names,
        RESNET_LAST_CONV_LAYER
    )
    print()


# DenseNet121 Grad-CAM
if densenet_model is not None:
    print("DenseNet121 Grad-CAM")
    visualize_gradcam(
        densenet_model,
        "DenseNet121",
        image_array,
        class_names,
        DENSENET_LAST_CONV_LAYER
    )
    print()


# CNN-XGBoost Grad-CAM
if cnn_xgb_model is not None:
    print("CNN-XGBoost Grad-CAM")
    visualize_gradcam(
        cnn_xgb_model,
        "CNN-XGBoost",
        image_array,
        class_names,
        CNN_XGB_LAST_CONV_LAYER
    )
    print()


# 7. Comparative Visualization

print("\n" + "="*70)
print("COMPARATIVE GRAD-CAM ANALYSIS")
print("="*70 + "\n")

# Prepare models dictionary
models_list = []
if resnet_model is not None:
    models_list.append({
        'name': 'ResNet50',
        'model': resnet_model,
        'layer': RESNET_LAST_CONV_LAYER
    })
if densenet_model is not None:
    models_list.append({
        'name': 'DenseNet121',
        'model': densenet_model,
        'layer': DENSENET_LAST_CONV_LAYER
    })
if cnn_xgb_model is not None:
    models_list.append({
        'name': 'CNN-XGBoost',
        'model': cnn_xgb_model,
        'layer': CNN_XGB_LAST_CONV_LAYER
    })

if len(models_list) > 0:
    compare_gradcam_all_models(models_list, image_array, class_names)


# 8. Multiple Images Visualization for Each Model

print("\n" + "="*70)
print("MULTIPLE IMAGES GRAD-CAM")
print("="*70 + "\n")

num_images_to_show = 6

if resnet_model is not None:
    print("ResNet50: Multiple Images")
    visualize_multiple_gradcam(
        resnet_model,
        "ResNet50",
        test_generator,
        class_names,
        RESNET_LAST_CONV_LAYER,
        num_images=num_images_to_show
    )
    print()

if densenet_model is not None:
    print("DenseNet121: Multiple Images")
    visualize_multiple_gradcam(
        densenet_model,
        "DenseNet121",
        test_generator,
        class_names,
        DENSENET_LAST_CONV_LAYER,
        num_images=num_images_to_show
    )
    print()

if cnn_xgb_model is not None:
    print("CNN-XGBoost: Multiple Images")
    visualize_multiple_gradcam(
        cnn_xgb_model,
        "CNN-XGBoost",
        test_generator,
        class_names,
        CNN_XGB_LAST_CONV_LAYER,
        num_images=num_images_to_show
    )
    print()

print("\n" + "="*70)
print("GRAD-CAM VISUALIZATION COMPLETE!")
print("="*70)


In [ ]:
# ============================================================
# MENINGIOMA-SPECIFIC GRAD-CAM ANALYSIS
# Extends the XAI comparison beyond glioma to meningioma, to
# visualize where the ResNet50 -> DenseNet121 / CNN-XGBoost
# improvement in meningioma classification actually occurs.
# ============================================================

import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tensorflow.keras.models import load_model

RESNET_LAST_CONV_LAYER   = "conv5_block3_out"
DENSENET_LAST_CONV_LAYER = "relu"
CNN_XGB_LAST_CONV_LAYER  = "activation_8"

def _safe_load(path, fallback):
    try:
        m = load_model(path)
        print(f"  Loaded {path}")
        return m
    except Exception:
        print(f"  {path} not found — using in-memory variable")
        return fallback

_models_gc = {
    'ResNet50'    : (_safe_load('best_resnet50_model.h5',   model),     RESNET_LAST_CONV_LAYER),
    'DenseNet121' : (_safe_load('best_densenet121_model.h5', model_dn), DENSENET_LAST_CONV_LAYER),
    'CNN-XGBoost' : (_safe_load('cnn_xgb_hybrid.h5',        cnn_model), CNN_XGB_LAST_CONV_LAYER),
}

# Collect ALL test images
test_generator.reset()
_X_all, _y_all = [], []
for _ in range(len(test_generator)):
    xb, yb = next(test_generator)
    _X_all.append(xb); _y_all.append(yb)
    if test_generator.batch_index == 0:
        break
_X_all = np.vstack(_X_all)[:test_generator.samples]
_y_all = np.vstack(_y_all)[:test_generator.samples]
_true_all = np.argmax(_y_all, axis=1)

meni_idx_gc = class_names.index('meningioma')
_meni_indices = np.where(_true_all == meni_idx_gc)[0]
print(f"\nFound {len(_meni_indices)} meningioma images in test set.")

_sample_indices = _meni_indices[:min(4, len(_meni_indices))]

def _gradcam(img_array, model, layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = int(tf.argmax(preds[0]))
        class_ch = preds[:, pred_index]
    grads  = tape.gradient(class_ch, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    hm     = conv_out[0] @ pooled[..., tf.newaxis]
    hm     = tf.squeeze(hm)
    hm     = tf.maximum(hm, 0)
    mx     = tf.reduce_max(hm)
    if mx != 0:
        hm /= mx
    return hm.numpy()

for sample_idx in _sample_indices:
    img_array = _X_all[sample_idx][np.newaxis, ...]
    fig = plt.figure(figsize=(20, 5 * len(_models_gc)))
    gs_main = gridspec.GridSpec(len(_models_gc), 4, figure=fig, hspace=0.4, wspace=0.3)

    for row_i, (mname, (mdl, layer)) in enumerate(_models_gc.items()):
        preds      = mdl.predict(img_array, verbose=0)
        pred_idx   = int(np.argmax(preds[0]))
        pred_label = class_names[pred_idx]
        confidence = preds[0][pred_idx]
        correct    = (pred_idx == meni_idx_gc)
        status     = "CORRECT ✓" if correct else f"WRONG → {pred_label}"
        bcol       = 'green' if correct else 'red'

        heatmap_raw = _gradcam(img_array, mdl, layer, pred_index=meni_idx_gc)

        # Brain mask for XAI (training images are already masked, this is extra safety)
        brain_mask  = create_brain_mask(img_array[0])
        heatmap_raw = apply_brain_mask_to_heatmap(heatmap_raw, brain_mask)

        hm_resized  = cv2.resize(heatmap_raw, (img_array.shape[2], img_array.shape[1]))
        hm_colored  = cv2.applyColorMap(np.uint8(255 * hm_resized), cv2.COLORMAP_JET)
        img_bgr     = cv2.cvtColor(np.uint8(img_array[0] * 255), cv2.COLOR_RGB2BGR)
        overlay     = cv2.addWeighted(img_bgr, 0.55, hm_colored, 0.45, 0)
        overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)

        ax1 = fig.add_subplot(gs_main[row_i, 0])
        ax1.imshow(img_array[0])
        ax1.set_title(f'{mname}\nTrue: meningioma\n{status}', fontweight='bold', fontsize=10, color=bcol)
        ax1.axis('off')

        ax2 = fig.add_subplot(gs_main[row_i, 1])
        im  = ax2.imshow(hm_resized, cmap='jet', vmin=0, vmax=1)
        ax2.set_title('Grad-CAM\n(meningioma target)', fontweight='bold', fontsize=10)
        ax2.axis('off')
        plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)

        ax3 = fig.add_subplot(gs_main[row_i, 2])
        ax3.imshow(overlay_rgb)
        ax3.set_title('Overlay (brain-masked)', fontweight='bold', fontsize=10)
        ax3.axis('off')

        ax4 = fig.add_subplot(gs_main[row_i, 3])
        bar_colors = ['green' if i == meni_idx_gc else 'orange' if i == pred_idx else 'lightgrey'
                      for i in range(len(class_names))]
        ax4.barh(class_names, preds[0] * 100, color=bar_colors)
        ax4.set_xlabel('Confidence (%)', fontsize=9)
        ax4.set_title('Class probabilities', fontweight='bold', fontsize=10)
        ax4.set_xlim(0, 100)
        ax4.axvline(50, color='grey', linestyle='--', linewidth=1)
        ax4.grid(axis='x', alpha=0.3)

    fig.suptitle(f'Meningioma Grad-CAM — Test Image #{sample_idx}\n'
                 'ResNet50 → DenseNet121 → CNN-XGBoost\n'
                 '(Green = meningioma | Brain-masked heatmaps suppress background)',
                 fontsize=12, fontweight='bold')
    plt.savefig(f'meningioma_gradcam_img{sample_idx}.png', dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Saved: meningioma_gradcam_img{sample_idx}.png")

In [ ]:
# INTEGRATED GRADIENTS ANALYSIS

import matplotlib.patches as patches
from scipy.ndimage import gaussian_filter
import pickle
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Configure GPU memory growth
try:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU memory growth enabled for {len(gpus)} GPU(s)")
except Exception as e:
    print(f"GPU configuration warning: {e}")

# Clear any existing GPU memory
try:
    tf.keras.backend.clear_session()
    print("Cleared TensorFlow session")
except:
    pass

print("="*80)
print("INTEGRATED GRADIENTS ANALYSIS")
print("="*80)

# LOAD MODELS AND DATA
print("\nLoading models and test data...")

# Try to load ResNet50
resnet_model = None
try:
    resnet_model = tf.keras.models.load_model('best_resnet50_model.h5')
    print("ResNet50 loaded from file")
except:
    try:
        if 'model' in dir() and isinstance(model, tf.keras.Model):
            resnet_model = model
            print("Using ResNet50 from memory (variable: model)")
    except:
        pass

# Try to load DenseNet121
densenet_model = None
try:
    densenet_model = tf.keras.models.load_model('best_densenet121_model.h5')
    print("DenseNet121 loaded from file")
except:
    try:
        if 'model_dn' in dir() and isinstance(model_dn, tf.keras.Model):
            densenet_model = model_dn
            print("Using DenseNet121 from memory (variable: model_dn)")
    except:
        pass

# Try to load CNN-XGBoost Hybrid
cnn_hybrid = None
try:
    cnn_hybrid = tf.keras.models.load_model('cnn_xgb_hybrid.h5')
    print("CNN-XGBoost loaded from file")
except:
    try:
        if 'cnn_model' in dir() and isinstance(cnn_model, tf.keras.Model):
            cnn_hybrid = cnn_model
            print("Using CNN-XGBoost from memory (variable: cnn_model)")
    except:
        pass

# Try to load XGBoost models
xgb_models = []
try:
    for i in range(1, 4):
        xgb_clf = xgb.XGBClassifier(n_jobs=-1)
        xgb_clf.load_model(f'xgb_model_{i}.json')
        xgb_models.append(xgb_clf)
    print(f"Loaded {len(xgb_models)} XGBoost models from files")
except:
    try:
        if 'models' in dir() and isinstance(models, list):
            xgb_models = [m for m in models if hasattr(m, 'predict_proba')]
            print(f"Using {len(xgb_models)} XGBoost models from memory (variable: models)")
    except:
        pass

try:
    with open('preprocessors.pkl', 'rb') as f:
        preprocessors = pickle.load(f)
    scaler = preprocessors['scaler']
    selector = preprocessors['selector']
    pca = preprocessors['pca']
    print("Preprocessors loaded")
except:
    print("Using preprocessors from memory")

test_generator.reset()
X_test_batch, y_test_batch = next(test_generator)
class_names = list(test_generator.class_indices.keys())

print(f"\nClass names: {class_names}")
print(f"Test batch shape: {X_test_batch.shape}")

# DIAGNOSTIC: Check model types
print("\n" + "="*60)
print("MODEL TYPE DIAGNOSTICS")
print("="*60)
print(f"ResNet model: {'✓ Loaded' if resnet_model is not None else '✗ Not available'}")
if resnet_model:
    print(f"  Type: {type(resnet_model)}")
print(f"DenseNet model: {'✓ Loaded' if densenet_model is not None else '✗ Not available'}")
if densenet_model:
    print(f"  Type: {type(densenet_model)}")
print(f"CNN Hybrid model: {'✓ Loaded' if cnn_hybrid is not None else '✗ Not available'}")
if cnn_hybrid:
    print(f"  Type: {type(cnn_hybrid)}")
print(f"XGBoost models: {'✓ Loaded' if len(xgb_models) > 0 else '✗ Not available'}")
if len(xgb_models) > 0:
    print(f"  Count: {len(xgb_models)}")
    print(f"  Type: {type(xgb_models[0])}")
print("="*60)

# Check if we have the minimum required models
if densenet_model is None and resnet_model is None:
    raise ValueError("No CNN models available! Please ensure models are loaded correctly.")
if cnn_hybrid is None or len(xgb_models) == 0:
    print("\nWARNING: CNN-XGBoost hybrid model not fully available. Will skip hybrid analysis.")

# Create feature extractor only if hybrid model is available
feature_extractor = None
if cnn_hybrid is not None:
    feature_extractor = tf.keras.Model(cnn_hybrid.input, cnn_hybrid.get_layer('features').output)

# SMART PREDICT FUNCTION (FIXED)
def safe_predict(model, data):
    """Predict safely for TensorFlow/Keras models only."""
    # Check if it's a Keras model
    if isinstance(model, tf.keras.Model):
        return model.predict(data, verbose=0)
    # Check if it has a predict method (functional API models)
    elif hasattr(model, 'predict') and callable(model.predict):
        try:
            return model.predict(data, verbose=0)
        except:
            return model.predict(data)
    else:
        print(f"Model type: {type(model)}")
        print(f"Model attributes: {dir(model)}")
        raise ValueError(f"Unsupported model type: {type(model)}. Only Keras models supported here.")

def predict_xgb_ensemble(xgb_models, features):
    """Predict using XGBoost ensemble on features."""
    probas = [m.predict_proba(features) for m in xgb_models]
    return np.mean(probas, axis=0)

# INTEGRATED GRADIENTS CORE FUNCTIONS
def compute_integrated_gradients(inputs, model, target_class_index, baseline=None, steps=100):
    """Compute Integrated Gradients with GPU memory safety."""
    if baseline is None:
        baseline = tf.zeros_like(inputs)

    # Convert to float32 to ensure compatibility
    inputs = tf.cast(inputs, tf.float32)
    baseline = tf.cast(baseline, tf.float32)

    alphas = tf.linspace(0.0, 1.0, steps + 1)
    input_diff = inputs - baseline

    # Process in smaller batches to avoid GPU memory issues
    batch_size = 10
    all_gradients = []

    for i in range(0, steps + 1, batch_size):
        end_idx = min(i + batch_size, steps + 1)
        batch_alphas = alphas[i:end_idx]

        interpolated_batch = baseline + batch_alphas[:, tf.newaxis, tf.newaxis, tf.newaxis] * input_diff

        with tf.GradientTape() as tape:
            tape.watch(interpolated_batch)
            preds = model(interpolated_batch, training=False)
            target_output = preds[:, target_class_index]

        gradients = tape.gradient(target_output, interpolated_batch)
        all_gradients.append(gradients.numpy())

        # Clear GPU cache after each batch
        tf.keras.backend.clear_session()

    # Combine all gradients
    all_gradients = np.concatenate(all_gradients, axis=0)
    avg_gradients = np.mean(all_gradients[:-1], axis=0)
    integrated_gradients = input_diff[0].numpy() * avg_gradients

    return integrated_gradients

def compute_multiple_baselines_ig(inputs, model, target_class_index, steps=50):
    """Compute IG with multiple baselines (reduced steps for memory efficiency)."""
    inputs_np = inputs.numpy() if isinstance(inputs, tf.Tensor) else inputs

    baselines = [
        tf.zeros_like(inputs),
        tf.ones_like(inputs),
        tf.random.uniform(inputs.shape, 0, 1),
        tf.ones_like(inputs) * 0.5,
        gaussian_filter(inputs_np[0], sigma=10)[np.newaxis, ...]
    ]

    attributions = []
    for i, baseline in enumerate(baselines):
        print(f"  Computing with baseline {i+1}/5...")
        baseline = tf.constant(baseline, dtype=tf.float32)
        attr = compute_integrated_gradients(inputs, model, target_class_index, baseline, steps)
        attributions.append(attr)

        # Clear memory after each baseline
        tf.keras.backend.clear_session()

    return np.mean(attributions, axis=0)

# VISUALIZATION FOR CNN MODELS (ResNet50, DenseNet121)
def visualize_integrated_gradients_cnn(model, model_name, img_array, class_names, img_idx=0, use_multiple_baselines=True):
    print(f"\n{'─'*60}")
    print(f"Computing Integrated Gradients for {model_name}")
    print(f"{'─'*60}")

    preds = safe_predict(model, img_array)
    pred_probs = preds[0]
    pred_index = np.argmax(pred_probs)
    predicted_class = class_names[pred_index]
    confidence = pred_probs[pred_index]

    true_label = np.argmax(y_test_batch[img_idx])
    true_class = class_names[true_label]

    print(f"True Label: {true_class}")
    print(f"Predicted: {predicted_class} (Confidence: {confidence:.2%})")

    if use_multiple_baselines:
        print("Computing IG with multiple baselines...")
        attribution = compute_multiple_baselines_ig(img_array, model, pred_index, steps=100)
    else:
        print("Computing IG with single baseline...")
        attribution = compute_integrated_gradients(img_array, model, pred_index, steps=100)

    abs_attr = np.sum(np.abs(attribution), axis=-1)
    abs_attr = abs_attr / (np.max(abs_attr) + 1e-10)

    pos_attr = np.maximum(0, np.sum(attribution, axis=-1))
    pos_attr = pos_attr / (np.max(pos_attr) + 1e-10)

    neg_attr = np.maximum(0, -np.sum(attribution, axis=-1))
    neg_attr = neg_attr / (np.max(neg_attr) + 1e-10)

    abs_attr_smooth = gaussian_filter(abs_attr, sigma=2)
    pos_attr_smooth = gaussian_filter(pos_attr, sigma=2)
    neg_attr_smooth = gaussian_filter(neg_attr, sigma=2)

    fig = plt.figure(figsize=(20, 10))
    ax1 = plt.subplot(2, 4, 1)
    ax1.imshow(img_array[0])
    ax1.set_title(f'Original Image\nTrue: {true_class}', fontweight='bold', fontsize=11)
    ax1.axis('off')
    border_color = 'green' if pred_index == true_label else 'red'
    rect = patches.Rectangle((0, 0), 1, 1, linewidth=4, edgecolor=border_color, facecolor='none', transform=ax1.transAxes)
    ax1.add_patch(rect)

    ax2 = plt.subplot(2, 4, 2)
    ax2.imshow(img_array[0], alpha=0.6)
    im2 = ax2.imshow(abs_attr_smooth, cmap='jet', alpha=0.6)
    ax2.set_title('Absolute Attribution (Total Influence)', fontweight='bold')
    ax2.axis('off')
    plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

    ax3 = plt.subplot(2, 4, 3)
    ax3.imshow(img_array[0], alpha=0.6)
    im3 = ax3.imshow(pos_attr_smooth, cmap='Greens', alpha=0.7)
    ax3.set_title('Positive Attribution (Supports Prediction)', color='green', fontweight='bold')
    ax3.axis('off')
    plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)

    ax4 = plt.subplot(2, 4, 4)
    ax4.imshow(img_array[0], alpha=0.6)
    im4 = ax4.imshow(neg_attr_smooth, cmap='Reds', alpha=0.7)
    ax4.set_title('Negative Attribution (Against Prediction)', color='red', fontweight='bold')
    ax4.axis('off')
    plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)

    fig.suptitle(f'Integrated Gradients: {model_name}\nPredicted: {predicted_class} ({confidence:.2%}) | True: {true_class}',
                 fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

    return attribution

# CNN–XGBOOST HYBRID MODEL (FIXED)

def extract_and_process_features(img_array, feature_extractor, preprocessors):
    """Extract CNN features and apply preprocessing pipeline."""
    features = feature_extractor.predict(img_array, verbose=0)

    # Feature engineering
    features_eng = np.column_stack([
        features,
        np.square(features),
        np.sqrt(np.abs(features) + 1e-8),
        np.log1p(np.abs(features)),
        features * np.roll(features, 1, axis=1)
    ])

    # Apply preprocessing pipeline
    features_scaled = preprocessors['scaler'].transform(features_eng)
    features_sel = preprocessors['selector'].transform(features_scaled)
    features_pca = preprocessors['pca'].transform(features_sel)

    return features_pca

def compute_ig_hybrid(inputs, feature_extractor, xgb_models, preprocessors, target_class_index, steps=30):
    """Compute Integrated Gradients for CNN-XGBoost hybrid using finite differences."""
    baseline = tf.zeros_like(inputs)
    alphas = tf.linspace(0.0, 1.0, steps + 1)
    input_diff = inputs - baseline

    # Convert to float32
    inputs = tf.cast(inputs, tf.float32)
    baseline = tf.cast(baseline, tf.float32)
    input_diff = tf.cast(input_diff, tf.float32)

    # We'll compute gradients through the CNN part only
    # Then use finite differences for the XGBoost impact

    all_gradients = []

    print(f"  Computing gradients for {steps+1} interpolation steps...")
    for i, alpha in enumerate(alphas):
        interpolated = baseline + alpha * input_diff

        with tf.GradientTape() as tape:
            tape.watch(interpolated)
            features = feature_extractor(interpolated, training=False)

        # Get gradient of features w.r.t. input
        feature_gradients = tape.gradient(features, interpolated)

        if feature_gradients is not None:
            # Get the prediction to weight the gradients
            features_np = features.numpy()

            # Feature engineering
            features_eng = np.column_stack([
                features_np,
                np.square(features_np),
                np.sqrt(np.abs(features_np) + 1e-8),
                np.log1p(np.abs(features_np)),
                features_np * np.roll(features_np, 1, axis=1)
            ])

            # Apply preprocessing
            features_scaled = preprocessors['scaler'].transform(features_eng)
            features_sel = preprocessors['selector'].transform(features_scaled)
            features_pca = preprocessors['pca'].transform(features_sel)

            # Get prediction probability as weight
            probas = [m.predict_proba(features_pca) for m in xgb_models]
            avg_proba = np.mean(probas, axis=0)
            weight = avg_proba[0, target_class_index]

            # Weight the gradients by prediction confidence
            weighted_gradient = feature_gradients.numpy() * weight
            all_gradients.append(weighted_gradient)

        # Progress indicator
        if (i + 1) % 10 == 0:
            print(f"    Progress: {i+1}/{steps+1}")

        # Clear memory periodically
        if (i + 1) % 5 == 0:
            tf.keras.backend.clear_session()

    # Average the gradients
    if len(all_gradients) > 0:
        all_gradients = np.array(all_gradients)
        avg_gradients = np.mean(all_gradients[:-1], axis=0)
        integrated_gradients = input_diff[0].numpy() * avg_gradients[0]
    else:
        print("  Warning: Could not compute gradients, using zero attribution")
        integrated_gradients = np.zeros_like(inputs[0].numpy())

    return integrated_gradients

def visualize_ig_hybrid(img_array, img_idx=0):
    """Visualize explanations for CNN-XGBoost hybrid model."""
    print(f"\n{'─'*60}")
    print(f"Computing Integrated Gradients for CNN-XGBoost Hybrid")
    print(f"{'─'*60}")

    # Get prediction
    features_pca = extract_and_process_features(img_array, feature_extractor,
                                                {'scaler': scaler, 'selector': selector, 'pca': pca})
    avg_proba = predict_xgb_ensemble(xgb_models, features_pca)[0]
    pred_index = np.argmax(avg_proba)
    predicted_class = class_names[pred_index]
    confidence = avg_proba[pred_index]

    true_label = np.argmax(y_test_batch[img_idx])
    true_class = class_names[true_label]

    print(f"True Label: {true_class}")
    print(f"Predicted: {predicted_class} (Confidence: {confidence:.2%})")

    try:
        print("Computing Integrated Gradients (CNN feature extraction)...")
        attribution = compute_ig_hybrid(img_array, feature_extractor, xgb_models,
                                        {'scaler': scaler, 'selector': selector, 'pca': pca},
                                        pred_index, steps=30)

        abs_attr = np.sum(np.abs(attribution), axis=-1)
        abs_attr = abs_attr / (np.max(abs_attr) + 1e-10)
        abs_attr_smooth = gaussian_filter(abs_attr, sigma=2)

        fig = plt.figure(figsize=(18, 6))

        ax1 = plt.subplot(1, 3, 1)
        ax1.imshow(img_array[0])
        ax1.axis('off')
        ax1.set_title(f'Original Image\nTrue: {true_class}', fontweight='bold')
        border_color = 'green' if pred_index == true_label else 'red'
        rect = patches.Rectangle((0, 0), 1, 1, linewidth=4, edgecolor=border_color, facecolor='none', transform=ax1.transAxes)
        ax1.add_patch(rect)

        ax2 = plt.subplot(1, 3, 2)
        ax2.imshow(img_array[0], alpha=0.6)
        im2 = ax2.imshow(abs_attr_smooth, cmap='jet', alpha=0.6)
        plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
        ax2.axis('off')
        ax2.set_title('Attribution Heatmap\n(CNN Feature Gradients)', fontweight='bold', fontsize=10)

        ax3 = plt.subplot(1, 3, 3)
        threshold = np.percentile(abs_attr, 95)
        mask = abs_attr > threshold
        masked_img = img_array[0].copy()
        masked_img[~mask] *= 0.3
        ax3.imshow(masked_img)
        ax3.axis('off')
        ax3.set_title(f'Top 5% Influential Regions\nThreshold: {threshold:.3f}', fontweight='bold', fontsize=10)

        fig.suptitle(f'Integrated Gradients: CNN-XGBoost Hybrid\nPredicted: {predicted_class} ({confidence:.2%}) | True: {true_class}',
                     fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()

        return attribution

    except Exception as e:
        print(f"  Error computing IG for hybrid model: {e}")
        print("  Falling back to simple visualization...")

        # Fallback: Just show the image with prediction
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        ax.imshow(img_array[0])
        ax.axis('off')
        ax.set_title(f'CNN-XGBoost Hybrid\nPredicted: {predicted_class} ({confidence:.2%})\nTrue: {true_class}',
                    fontweight='bold')
        border_color = 'green' if pred_index == true_label else 'red'
        rect = patches.Rectangle((0, 0), 1, 1, linewidth=4, edgecolor=border_color, facecolor='none', transform=ax.transAxes)
        ax.add_patch(rect)
        plt.tight_layout()
        plt.show()

        return None

# EXECUTE ANALYSIS
print("\nGenerating Integrated Gradients explanations...")

# Use meningioma test images instead of arbitrary fixed indices,
# so the reported figure reliably explains the hardest class
# rather than whatever happened to land at positions 0, 5, 10.
meni_idx_ig = class_names.index('meningioma')
_batch_true_ig = np.argmax(y_test_batch, axis=1)
_meni_in_batch_ig = np.where(_batch_true_ig == meni_idx_ig)[0]

if len(_meni_in_batch_ig) > 0:
    image_indices = _meni_in_batch_ig[:3].tolist()
    print(f"Using meningioma image indices from current batch: {image_indices}")
else:
    test_generator.reset()
    _X_all_ig, _y_all_ig = [], []
    for _ in range(len(test_generator)):
        xb, yb = next(test_generator)
        _X_all_ig.append(xb); _y_all_ig.append(yb)
        if test_generator.batch_index == 0:
            break
    _X_all_ig = np.vstack(_X_all_ig)[:test_generator.samples]
    _y_all_ig = np.vstack(_y_all_ig)[:test_generator.samples]
    _true_all_ig = np.argmax(_y_all_ig, axis=1)
    _meni_full_ig = np.where(_true_all_ig == meni_idx_ig)[0]
    print("No meningioma image in current batch; substituting from the full test set.")
    X_test_batch = _X_all_ig
    y_test_batch = _y_all_ig
    image_indices = _meni_full_ig[:3].tolist()

for img_idx in image_indices:
    print(f"\n{'='*80}")
    print(f"ANALYZING IMAGE #{img_idx}")
    print(f"{'='*80}")

    img_array = X_test_batch[img_idx][np.newaxis, ...]

    # Analyze with ResNet if available
    if resnet_model is not None:
        visualize_integrated_gradients_cnn(resnet_model, "ResNet50", img_array, class_names, img_idx)

    # Analyze with DenseNet if available
    if densenet_model is not None:
        visualize_integrated_gradients_cnn(densenet_model, "DenseNet121", img_array, class_names, img_idx)

    # Analyze with hybrid model if available
    if cnn_hybrid is not None and len(xgb_models) > 0 and feature_extractor is not None:
        visualize_ig_hybrid(img_array, img_idx)
    else:
        print(f"\nSkipping CNN-XGBoost hybrid analysis (model not available)")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)

In [ ]:
# INTEGRATED GRADIENTS - MENINGIOMA-SPECIFIC ANALYSIS
import gc
from tensorflow.keras.models import load_model
import warnings
warnings.filterwarnings('ignore')

# Free any models/graphs left over from previous runs of this cell
try:
    tf.keras.backend.clear_session()
except:
    pass
gc.collect()

try:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision enabled")
except:
    print("Mixed precision not available")

print("MENINGIOMA-SPECIFIC INTEGRATED GRADIENTS ANALYSIS\n" + "="*60)


# LOAD MODELS WITH ERROR HANDLING
print("Loading models...")

try:
    resnet_model = load_model('best_resnet50_model.h5', compile=False)
    print("ResNet50 loaded")
except:
    try:
        resnet_model = model
        print("ResNet50 from memory")
    except:
        print("ResNet50 not found - skipping")
        resnet_model = None

try:
    densenet_model = load_model('best_densenet121_model.h5', compile=False)
    print("DenseNet121 loaded")
except:
    try:
        densenet_model = model_dn
        print("DenseNet121 from memory")
    except:
        print("DenseNet121 not found - skipping")
        densenet_model = None

try:
    cnn_hybrid = load_model('cnn_xgb_hybrid.h5', compile=False)
    print("CNN-XGBoost loaded")
except:
    try:
        cnn_hybrid = cnn_model
        print("CNN-XGBoost from memory")
    except:
        print("CNN-XGBoost not found - skipping")
        cnn_hybrid = None


# STREAM THE TEST SET TO FIND THE FIRST MENINGIOMA IMAGE
# No correctness filtering - fairly (not cherry-picked) selected,
# same image explained across every model.
print("\nSearching test set for a meningioma image...")
test_generator.reset()
class_names = list(test_generator.class_indices.keys())
meni_label_idx = class_names.index('meningioma')

img = None
true_label = None

n_batches = len(test_generator)
for b in range(n_batches):
    xb, yb = next(test_generator)
    true_labels = np.argmax(yb, axis=1)
    meni_positions = np.where(true_labels == meni_label_idx)[0]

    if len(meni_positions) > 0:
        img = xb[meni_positions[0]].copy()
        true_label = meni_label_idx
        del xb, yb
        break

    del xb, yb

gc.collect()

if img is None:
    raise RuntimeError("No meningioma image found in test set - check class_names / generator.")

print(f"Using the first meningioma image found (batch {b}).")
print()


# INTEGRATED GRADIENTS - computed in small interpolation-step
# mini-batches so the full (steps, H, W, C) tensor is never held
# in memory at once. 50 steps
# as a memory/quality tradeoff - raise back to 100 if RAM allows.
IG_STEPS = 50
IG_BATCH = 10  # interpolation steps processed per forward pass

def integrated_gradients(model, img, pred_class, steps=IG_STEPS, mini_batch=IG_BATCH):
    """Integrated Gradients w.r.t. a zero baseline, computed in
    small mini-batches of interpolation steps to bound peak memory."""
    baseline = np.zeros_like(img)
    alphas = np.linspace(0.0, 1.0, steps + 1)

    accumulated_grads = np.zeros_like(img, dtype=np.float32)

    for start in range(0, len(alphas), mini_batch):
        chunk_alphas = alphas[start:start + mini_batch]
        # interpolated images for this chunk only
        interpolated = baseline[np.newaxis, ...] + \
            chunk_alphas[:, np.newaxis, np.newaxis, np.newaxis] * \
            (img[np.newaxis, ...] - baseline[np.newaxis, ...])
        interpolated = tf.convert_to_tensor(interpolated, dtype=tf.float32)

        with tf.GradientTape() as tape:
            tape.watch(interpolated)
            preds = model(interpolated, training=False)
            target_preds = preds[:, pred_class]

        grads = tape.gradient(target_preds, interpolated)
        accumulated_grads += tf.reduce_sum(grads, axis=0).numpy()

        del interpolated, preds, target_preds, grads
        gc.collect()

    avg_grads = accumulated_grads / steps
    ig = (img - baseline) * avg_grads
    return ig  # (H, W, C), signed attribution


models_to_explain = [
    ("ResNet50", resnet_model),
    ("DenseNet121", densenet_model),
    ("CNN-XGBoost", cnn_hybrid),
]

print("Integrated Gradients Models (meningioma):\n" + "-"*60)

ig_maps = {}
pred_info = {}

for name, model in models_to_explain:
    if model is None:
        print(f"⊗ {name} skipped (model not loaded)")
        continue

    try:
        pred_proba = model.predict(img[np.newaxis, ...], verbose=0)[0]
        pred_class = np.argmax(pred_proba)
        print(f"→ {name}: True={class_names[true_label]}, Pred={class_names[pred_class]} ({pred_proba[pred_class]:.1%})")

        ig = integrated_gradients(model, img, pred_class)

        # Reduce to single-channel maps, then brain-mask each
        _brain_mask = create_brain_mask(img)

        total_attr = apply_brain_mask_to_heatmap(np.sum(np.abs(ig), axis=-1), _brain_mask)
        pos_attr = apply_brain_mask_to_heatmap(np.sum(np.clip(ig, 0, None), axis=-1), _brain_mask)
        neg_attr = apply_brain_mask_to_heatmap(np.sum(np.clip(-ig, 0, None), axis=-1), _brain_mask)

        ig_maps[name] = (total_attr, pos_attr, neg_attr)
        pred_info[name] = (pred_class, pred_proba[pred_class])

        del ig
        gc.collect()

        print(f"{name} Integrated Gradients complete")

    except Exception as e:
        print(f"Error explaining {name}: {e}")

print()

# Comparison figure - one row per model: original | total | positive | negative
n_rows = len(ig_maps)
if n_rows > 0:
    fig, axes = plt.subplots(n_rows, 4, figsize=(16, 4 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    for row, (name, (total_attr, pos_attr, neg_attr)) in enumerate(ig_maps.items()):
        pred_class, conf = pred_info[name]

        axes[row, 0].imshow(img)
        axes[row, 0].set_title(f'{name}\nOriginal (True: {class_names[true_label]})', fontweight='bold')
        axes[row, 0].axis('off')

        im1 = axes[row, 1].imshow(total_attr, cmap='hot')
        axes[row, 1].set_title(f'Absolute Attribution\nPred: {class_names[pred_class]} ({conf:.1%})', fontweight='bold')
        axes[row, 1].axis('off')
        plt.colorbar(im1, ax=axes[row, 1], fraction=0.046, pad=0.04)

        im2 = axes[row, 2].imshow(pos_attr, cmap='Greens')
        axes[row, 2].set_title('Positive Attribution\n(Supports Prediction)', fontweight='bold')
        axes[row, 2].axis('off')
        plt.colorbar(im2, ax=axes[row, 2], fraction=0.046, pad=0.04)

        im3 = axes[row, 3].imshow(neg_attr, cmap='Reds')
        axes[row, 3].set_title('Negative Attribution\n(Against Prediction)', fontweight='bold')
        axes[row, 3].axis('off')
        plt.colorbar(im3, ax=axes[row, 3], fraction=0.046, pad=0.04)

    plt.suptitle('Integrated Gradients Analysis (Meningioma)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

del ig_maps
gc.collect()

print("="*60)
print("MENINGIOMA INTEGRATED GRADIENTS ANALYSIS COMPLETE")
print("="*60)

In [ ]:
# LIME
!pip install lime
!pip install lime scikit-image -q

from tensorflow.keras.models import load_model
from lime import lime_image, lime_tabular
from skimage.segmentation import mark_boundaries
import pickle
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Enable mixed precision for speed
try:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision enabled")
except:
    print("Mixed precision not available")

print("FAST LIME ANALYSIS\n" + "="*60)


# LOAD MODELS WITH ERROR HANDLING
print("Loading models...")

try:
    resnet_model = load_model('best_resnet50_model.h5', compile=False)
    print("ResNet50 loaded")
except:
    try:
        resnet_model = model
        print("ResNet50 from memory")
    except:
        print("ResNet50 not found - skipping")
        resnet_model = None

try:
    densenet_model = load_model('best_densenet121_model.h5', compile=False)
    print("DenseNet121 loaded")
except:
    try:
        densenet_model = model_dn
        print("DenseNet121 from memory")
    except:
        print("DenseNet121 not found - skipping")
        densenet_model = None

try:
    cnn_hybrid = load_model('cnn_xgb_hybrid.h5', compile=False)
    print("CNN-XGBoost loaded")
except:
    try:
        cnn_hybrid = cnn_model
        print("CNN-XGBoost from memory")
    except:
        print("CNN-XGBoost not found - skipping")
        cnn_hybrid = None

# Load XGBoost models
xgb_models = []
try:
    for i in range(1, 4):
        xgb_clf = xgb.XGBClassifier(n_jobs=-1)
        xgb_clf.load_model(f'xgb_model_{i}.json')
        xgb_models.append(xgb_clf)
    print(f"Loaded {len(xgb_models)} XGBoost models")
except:
    try:
        xgb_models = models
        print(f"XGBoost models from memory ({len(xgb_models)} models)")
    except:
        print("XGBoost models not found")
        xgb_models = []

# Load preprocessors
try:
    with open('preprocessors.pkl', 'rb') as f:
        preprocessors = pickle.load(f)
    scaler = preprocessors['scaler']
    selector = preprocessors['selector']
    pca = preprocessors['pca']
    print("Preprocessors loaded")
except:
    try:
        # Assume they exist in memory
        print("Preprocessors from memory")
    except:
        print("Preprocessors not found")
        scaler = selector = pca = None

# Get test data
try:
    test_generator.reset()
    X_test_batch, y_test_batch = next(test_generator)
    class_names = list(test_generator.class_indices.keys())
    print(f"Test data loaded | Classes: {class_names}")
except Exception as e:
    print(f"Error loading test data: {e}")
    raise

print()

# CNN LIME (ResNet & DenseNet)
if resnet_model is not None or densenet_model is not None:
    print("CNN Models LIME:\n" + "-"*60)

    explainer_image = lime_image.LimeImageExplainer(random_state=42)

    def explain_cnn(model, name, img_idx=0):
        """Fast LIME for CNN models"""
        if model is None:
            print(f"⊗ {name} skipped (model not loaded)")
            return

        try:
            img = X_test_batch[img_idx]
            true_label = np.argmax(y_test_batch[img_idx])

            # Get prediction
            pred_proba = model.predict(img[np.newaxis, ...], verbose=0)[0]
            pred_class = np.argmax(pred_proba)

            print(f"→ {name}: True={class_names[true_label]}, Pred={class_names[pred_class]} ({pred_proba[pred_class]:.1%})")

            # Generate LIME explanation
            explanation = explainer_image.explain_instance(
                img,
                lambda x: model.predict(np.clip(x, 0, 1), verbose=0, batch_size=128),
                top_labels=1,
                hide_color=0,
                num_samples=500,
                batch_size=128
            )

            # Visualize
            fig, axes = plt.subplots(1, 3, figsize=(12, 4))

            # Original
            axes[0].imshow(img)
            axes[0].set_title(f'Original\n{class_names[true_label]}', fontweight='bold')
            axes[0].axis('off')

            # Positive evidence
            temp_pos, mask_pos = explanation.get_image_and_mask(
                pred_class, positive_only=True, num_features=8, hide_rest=False
            )
            axes[1].imshow(mark_boundaries(temp_pos, mask_pos, color=(0,1,0)))
            axes[1].set_title('Supporting Evidence', fontweight='bold', color='green')
            axes[1].axis('off')

            # All important regions
            temp_all, mask_all = explanation.get_image_and_mask(
                pred_class, positive_only=False, num_features=10, hide_rest=True
            )
            axes[2].imshow(mark_boundaries(img, mask_all, color=(1,1,0)))
            axes[2].set_title('Key Regions', fontweight='bold', color='orange')
            axes[2].axis('off')

            plt.suptitle(f'{name}: {class_names[pred_class]} ({pred_proba[pred_class]:.1%})',
                        fontsize=13, fontweight='bold')
            plt.tight_layout()
            plt.show()

            print(f"{name} explanation complete\n")

        except Exception as e:
            print(f"Error explaining {name}: {e}\n")

    # Run CNN explanations
    explain_cnn(resnet_model, "ResNet50", 0)
    explain_cnn(densenet_model, "DenseNet121", 0)

else:
    print("CNN models not available - skipping\n")

# HYBRID LIME (CNN-XGBoost)
if cnn_hybrid is not None and len(xgb_models) > 0 and scaler is not None:
    print("CNN-XGBoost Hybrid LIME:\n" + "-"*60)

    try:
        # Pre-compute features
        print("→ Extracting features...")
        feature_extractor = tf.keras.Model(
            cnn_hybrid.input,
            cnn_hybrid.get_layer('features').output
        )
        X_test_features = feature_extractor.predict(X_test_batch, verbose=0, batch_size=32)

        # Feature engineering
        X_test_eng = np.column_stack([
            X_test_features,
            np.square(X_test_features),
            np.sqrt(np.abs(X_test_features) + 1e-8),
            np.log1p(np.abs(X_test_features)),
            X_test_features * np.roll(X_test_features, 1, axis=1)
        ])

        # Apply preprocessing
        X_test_scaled = scaler.transform(X_test_eng)
        X_test_sel = selector.transform(X_test_scaled)
        X_test_pca_batch = pca.transform(X_test_sel)

        print(f"Features ready: {X_test_pca_batch.shape}")

        # Ensemble prediction function
        def ensemble_predict(data):
            probas = [m.predict_proba(data) for m in xgb_models]
            return np.mean(probas, axis=0)

        # Initialize LIME explainer
        feature_names = [f'PC_{i+1}' for i in range(X_test_pca_batch.shape[1])]
        explainer_tabular = lime_tabular.LimeTabularExplainer(
            X_test_pca_batch,
            feature_names=feature_names,
            class_names=class_names,
            mode='classification',
            random_state=42
        )

        def explain_hybrid(idx=0):
            """Fast LIME for hybrid model"""
            try:
                instance = X_test_pca_batch[idx]
                true_label = np.argmax(y_test_batch[idx])

                # Get prediction
                pred_proba = ensemble_predict(instance[np.newaxis, ...])[0]
                pred_class = np.argmax(pred_proba)

                print(f"→ Hybrid: True={class_names[true_label]}, Pred={class_names[pred_class]} ({pred_proba[pred_class]:.1%})")

                # Generate explanation
                explanation = explainer_tabular.explain_instance(
                    instance,
                    ensemble_predict,
                    num_features=10,
                    top_labels=1,
                    num_samples=1000
                )

                # Visualize
                fig, axes = plt.subplots(1, 2, figsize=(12, 4))

                # Feature contributions
                exp_list = explanation.as_list(label=pred_class)
                features = [item[0] for item in exp_list]
                weights = [item[1] for item in exp_list]
                colors = ['green' if w > 0 else 'red' for w in weights]

                axes[0].barh(range(len(features)), weights, color=colors, alpha=0.7)
                axes[0].set_yticks(range(len(features)))
                axes[0].set_yticklabels(features, fontsize=8)
                axes[0].set_xlabel('Weight', fontweight='bold')
                axes[0].set_title('Feature Contributions', fontweight='bold')
                axes[0].axvline(0, color='black', linewidth=0.8)
                axes[0].grid(axis='x', alpha=0.3)

                # Prediction probabilities
                bar_colors = ['green' if i==pred_class else 'gray' for i in range(len(class_names))]
                bars = axes[1].bar(class_names, pred_proba*100, color=bar_colors, alpha=0.7)
                axes[1].set_ylabel('Probability (%)', fontweight='bold')
                axes[1].set_title('Class Probabilities', fontweight='bold')
                axes[1].set_ylim([0, 100])

                for bar, prob in zip(bars, pred_proba):
                    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                               f'{prob:.1%}', ha='center', va='bottom', fontsize=9)

                plt.suptitle(f'Hybrid: {class_names[pred_class]} ({pred_proba[pred_class]:.1%})',
                           fontsize=13, fontweight='bold')
                plt.tight_layout()
                plt.show()

                print(f"Hybrid explanation complete\n")

            except Exception as e:
                print(f"Error in hybrid explanation: {e}\n")

        # Run hybrid explanation
        explain_hybrid(0)

    except Exception as e:
        print(f"Error in hybrid setup: {e}\n")

else:
    print("Hybrid model not available - skipping\n")

print("="*60)
print("LIME ANALYSIS COMPLETE")
print("="*60)

In [ ]:
# LIME - MENINGIOMA-SPECIFIC ANALYSIS (unbiased selection)
!pip install lime
!pip install lime scikit-image -q

import gc
from tensorflow.keras.models import load_model
from lime import lime_image, lime_tabular
from skimage.segmentation import mark_boundaries
import pickle
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Free any models/graphs left over from previous runs of this cell
try:
    tf.keras.backend.clear_session()
except:
    pass
gc.collect()

try:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision enabled")
except:
    print("Mixed precision not available")

print("MENINGIOMA-SPECIFIC LIME ANALYSIS\n" + "="*60)


# LOAD MODELS WITH ERROR HANDLING
print("Loading models...")

try:
    resnet_model = load_model('best_resnet50_model.h5', compile=False)
    print("ResNet50 loaded")
except:
    try:
        resnet_model = model
        print("ResNet50 from memory")
    except:
        print("ResNet50 not found - skipping")
        resnet_model = None

try:
    densenet_model = load_model('best_densenet121_model.h5', compile=False)
    print("DenseNet121 loaded")
except:
    try:
        densenet_model = model_dn
        print("DenseNet121 from memory")
    except:
        print("DenseNet121 not found - skipping")
        densenet_model = None

try:
    cnn_hybrid = load_model('cnn_xgb_hybrid.h5', compile=False)
    print("CNN-XGBoost loaded")
except:
    try:
        cnn_hybrid = cnn_model
        print("CNN-XGBoost from memory")
    except:
        print("CNN-XGBoost not found - skipping")
        cnn_hybrid = None

xgb_models = []
try:
    for i in range(1, 4):
        xgb_clf = xgb.XGBClassifier(n_jobs=-1)
        xgb_clf.load_model(f'xgb_model_{i}.json')
        xgb_models.append(xgb_clf)
    print(f"Loaded {len(xgb_models)} XGBoost models")
except:
    try:
        xgb_models = models
        print(f"XGBoost models from memory ({len(xgb_models)} models)")
    except:
        print("XGBoost models not found")
        xgb_models = []

try:
    with open('preprocessors.pkl', 'rb') as f:
        preprocessors = pickle.load(f)
    scaler = preprocessors['scaler']
    selector = preprocessors['selector']
    pca = preprocessors['pca']
    print("Preprocessors loaded")
except:
    try:
        print("Preprocessors from memory")
    except:
        print("Preprocessors not found")
        scaler = selector = pca = None


# STREAM THE TEST SET TO FIND THE FIRST MENINGIOMA IMAGE
# No correctness filtering - this is just the first meningioma
# image encountered, so every model is explained on the same,
# fairly (not cherry-picked) selected case.
print("\nSearching test set for a meningioma image...")
test_generator.reset()
class_names = list(test_generator.class_indices.keys())
meni_label_idx = class_names.index('meningioma')

img = None
true_label = None

n_batches = len(test_generator)
for b in range(n_batches):
    xb, yb = next(test_generator)
    true_labels = np.argmax(yb, axis=1)
    meni_positions = np.where(true_labels == meni_label_idx)[0]

    if len(meni_positions) > 0:
        img = xb[meni_positions[0]].copy()
        true_label = meni_label_idx
        del xb, yb
        break

    del xb, yb

gc.collect()

if img is None:
    raise RuntimeError("No meningioma image found in test set - check class_names / generator.")

print(f"Using the first meningioma image found (batch {b}).")
print()


# CNN LIME (ResNet & DenseNet) - single meningioma image, no batch stored
if resnet_model is not None or densenet_model is not None:
    print("CNN Models LIME (meningioma):\n" + "-"*60)

    explainer_image = lime_image.LimeImageExplainer(random_state=42)

    def explain_cnn(model, name, img, true_label):
        """LIME for CNN models on a single pre-selected image."""
        if model is None:
            print(f"⊗ {name} skipped (model not loaded)")
            return

        try:
            pred_proba = model.predict(img[np.newaxis, ...], verbose=0)[0]
            pred_class = np.argmax(pred_proba)

            print(f"→ {name}: True={class_names[true_label]}, Pred={class_names[pred_class]} ({pred_proba[pred_class]:.1%})")

            explanation = explainer_image.explain_instance(
                img,
                lambda x: model.predict(np.clip(x, 0, 1), verbose=0, batch_size=64),
                top_labels=1,
                hide_color=0,
                num_samples=300,   # reduced from 500 to cut peak memory
                batch_size=64
            )

            fig, axes = plt.subplots(1, 3, figsize=(12, 4))

            axes[0].imshow(img)
            axes[0].set_title(f'Original\n{class_names[true_label]}', fontweight='bold')
            axes[0].axis('off')

            temp_pos, mask_pos = explanation.get_image_and_mask(
                pred_class, positive_only=True, num_features=8, hide_rest=False
            )
            axes[1].imshow(mark_boundaries(temp_pos, mask_pos, color=(0,1,0)))
            axes[1].set_title('Supporting Evidence', fontweight='bold', color='green')
            axes[1].axis('off')

            temp_all, mask_all = explanation.get_image_and_mask(
                pred_class, positive_only=False, num_features=10, hide_rest=True
            )
            axes[2].imshow(mark_boundaries(img, mask_all, color=(1,1,0)))
            axes[2].set_title('Key Regions', fontweight='bold', color='orange')
            axes[2].axis('off')

            plt.suptitle(f'{name}: {class_names[pred_class]} ({pred_proba[pred_class]:.1%})',
                        fontsize=13, fontweight='bold')
            plt.tight_layout()
            plt.show()

            del explanation, temp_pos, mask_pos, temp_all, mask_all
            gc.collect()

            print(f"{name} explanation complete\n")

        except Exception as e:
            print(f"Error explaining {name}: {e}\n")

    explain_cnn(resnet_model, "ResNet50", img, true_label)
    explain_cnn(densenet_model, "DenseNet121", img, true_label)

else:
    print("CNN models not available - skipping\n")

# HYBRID LIME (CNN-XGBoost) - features extracted for ONE image + a
# small background sample (not the full 2146-image test set)
if cnn_hybrid is not None and len(xgb_models) > 0 and scaler is not None:
    print("CNN-XGBoost Hybrid LIME (meningioma):\n" + "-"*60)

    try:
        print("→ Extracting features (target image + small background sample)...")
        feature_extractor = tf.keras.Model(
            cnn_hybrid.input,
            cnn_hybrid.get_layer('features').output
        )

        # Small background sample for LIME's tabular perturbation
        # statistics (needs *some* distribution, not the whole test set).
        test_generator.reset()
        BACKGROUND_SIZE = 150
        bg_imgs = []
        collected = 0
        for _ in range(len(test_generator)):
            xb, yb = next(test_generator)
            take = min(len(xb), BACKGROUND_SIZE - collected)
            if take > 0:
                bg_imgs.append(xb[:take])
                collected += take
            del xb, yb
            if collected >= BACKGROUND_SIZE:
                break
        bg_imgs = np.vstack(bg_imgs)
        # make sure the target image itself is included
        combined_imgs = np.vstack([img[np.newaxis, ...], bg_imgs])

        combined_features = feature_extractor.predict(combined_imgs, verbose=0, batch_size=32)
        del bg_imgs, combined_imgs
        gc.collect()

        combined_eng = np.column_stack([
            combined_features,
            np.square(combined_features),
            np.sqrt(np.abs(combined_features) + 1e-8),
            np.log1p(np.abs(combined_features)),
            combined_features * np.roll(combined_features, 1, axis=1)
        ])
        del combined_features
        gc.collect()

        combined_scaled = scaler.transform(combined_eng)
        combined_sel = selector.transform(combined_scaled)
        combined_pca = pca.transform(combined_sel)
        del combined_eng, combined_scaled, combined_sel
        gc.collect()

        target_pca = combined_pca[0]           # the meningioma image
        background_pca = combined_pca[1:]      # background for LIME stats

        print(f"Target features ready: {target_pca.shape} | Background: {background_pca.shape}")

        def ensemble_predict(data):
            probas = [m.predict_proba(data) for m in xgb_models]
            return np.mean(probas, axis=0)

        feature_names = [f'PC_{i+1}' for i in range(background_pca.shape[1])]
        explainer_tabular = lime_tabular.LimeTabularExplainer(
            background_pca,
            feature_names=feature_names,
            class_names=class_names,
            mode='classification',
            random_state=42
        )

        def explain_hybrid(instance, true_label):
            """LIME for hybrid model on a single pre-computed PCA instance."""
            try:
                pred_proba = ensemble_predict(instance[np.newaxis, ...])[0]
                pred_class = np.argmax(pred_proba)

                print(f"→ Hybrid: True={class_names[true_label]}, Pred={class_names[pred_class]} ({pred_proba[pred_class]:.1%})")

                explanation = explainer_tabular.explain_instance(
                    instance,
                    ensemble_predict,
                    num_features=10,
                    top_labels=1,
                    num_samples=500  # reduced from 1000
                )

                fig, axes = plt.subplots(1, 2, figsize=(12, 4))

                exp_list = explanation.as_list(label=pred_class)
                features = [item[0] for item in exp_list]
                weights = [item[1] for item in exp_list]
                colors = ['green' if w > 0 else 'red' for w in weights]

                axes[0].barh(range(len(features)), weights, color=colors, alpha=0.7)
                axes[0].set_yticks(range(len(features)))
                axes[0].set_yticklabels(features, fontsize=8)
                axes[0].set_xlabel('Weight', fontweight='bold')
                axes[0].set_title('Feature Contributions', fontweight='bold')
                axes[0].axvline(0, color='black', linewidth=0.8)
                axes[0].grid(axis='x', alpha=0.3)

                bar_colors = ['green' if i==pred_class else 'gray' for i in range(len(class_names))]
                bars = axes[1].bar(class_names, pred_proba*100, color=bar_colors, alpha=0.7)
                axes[1].set_ylabel('Probability (%)', fontweight='bold')
                axes[1].set_title('Class Probabilities', fontweight='bold')
                axes[1].set_ylim([0, 100])

                for bar, prob in zip(bars, pred_proba):
                    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                               f'{prob:.1%}', ha='center', va='bottom', fontsize=9)

                plt.suptitle(f'Hybrid: {class_names[pred_class]} ({pred_proba[pred_class]:.1%})',
                           fontsize=13, fontweight='bold')
                plt.tight_layout()
                plt.show()

                print(f"Hybrid explanation complete\n")

            except Exception as e:
                print(f"Error in hybrid explanation: {e}\n")

        explain_hybrid(target_pca, true_label)
        del background_pca, combined_pca
        gc.collect()

    except Exception as e:
        print(f"Error in hybrid setup: {e}\n")

else:
    print("Hybrid model not available - skipping\n")

print("="*60)
print("MENINGIOMA LIME ANALYSIS COMPLETE")
print("="*60)

In [ ]:
#Comprehensive XAI Evaluation Framework
#Evaluates Grad-CAM, Integrated Gradients, LIME, and SHAP on CNN and Hybrid models

from scipy.stats import spearmanr
from scipy.ndimage import gaussian_filter
import cv2
import pickle
import xgboost as xgb
from lime import lime_image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import shap
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
CONV_LAYERS = {'ResNet50': 'conv5_block3_out', 'DenseNet121': 'relu', 'CNN-XGBoost': 'global_average_pooling2d'}
EVAL_IMAGES = [0, 3, 5, 8, 10]
BACKGROUND_SAMPLES = 20

# MODEL LOADING
def load_models():
    """Load all models and preprocessors"""
    models = {}

    # CNN Models
    for name, path in [('DenseNet121', 'best_densenet121_model.h5'),
                       ('ResNet50', 'best_resnet50_model.h5'),
                       ('CNN-XGBoost', 'cnn_xgb_hybrid.h5')]:
        try:
            models[name] = tf.keras.models.load_model(path)
            print(f"✓ Loaded {name}")
        except Exception as e:
            models[name] = None
            print(f"✗ Failed to load {name}: {str(e)[:50]}")

    # XGBoost Models
    models['XGB'] = []
    try:
        for i in range(1, 4):
            xgb_clf = xgb.XGBClassifier(n_jobs=-1)
            xgb_clf.load_model(f'xgb_model_{i}.json')
            models['XGB'].append(xgb_clf)
        print(f"✓ Loaded {len(models['XGB'])} XGBoost models")
    except Exception as e:
        print(f"✗ Failed to load XGBoost: {str(e)[:50]}")

    # Preprocessors
    try:
        with open('preprocessors.pkl', 'rb') as f:
            models['prep'] = pickle.load(f)
        print("✓ Loaded preprocessors")
    except:
        models['prep'] = None

    return models

# XAI METHODS
def gradcam(img, model, layer_name, pred_idx):
    """Generate Grad-CAM heatmap"""
    grad_model = tf.keras.Model([model.inputs],
                                 [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img)
        loss = preds[:, pred_idx]

    grads = tape.gradient(loss, conv_out)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(conv_out[0] * weights, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-10)
    return cv2.resize(heatmap.numpy(), (img.shape[2], img.shape[1]))

def integrated_gradients(img, model, pred_idx, steps=50):
    """Compute Integrated Gradients"""
    baseline = tf.zeros_like(img)
    alphas = tf.linspace(0.0, 1.0, steps + 1)
    interpolated = baseline + alphas[:, None, None, None] * (img - baseline)

    with tf.GradientTape() as tape:
        tape.watch(interpolated)
        preds = model(interpolated, training=False)[:, pred_idx]

    grads = tape.gradient(preds, interpolated)
    ig = (img - baseline)[0] * tf.reduce_mean(grads[:-1], axis=0)
    heatmap = np.sum(np.abs(ig.numpy()), axis=-1)
    return heatmap / (heatmap.max() + 1e-10)

def lime_explain(img, model, pred_idx):
    """Generate LIME explanation"""
    explainer = lime_image.LimeImageExplainer(random_state=42)
    exp = explainer.explain_instance(img[0], lambda x: model.predict(x, verbose=0),
                                     top_labels=1, num_samples=500, batch_size=50)
    _, mask = exp.get_image_and_mask(pred_idx, positive_only=False, num_features=10)
    heatmap = np.zeros(img.shape[1:3])
    for seg, weight in exp.local_exp[pred_idx]:
        heatmap[mask == seg] = abs(weight)
    return heatmap / (heatmap.max() + 1e-10)

def shap_explain(img, model, background, pred_idx):
    """Generate SHAP explanation"""
    explainer = shap.DeepExplainer(model, background)
    shap_values = explainer.shap_values(img)
    shap_img = shap_values[pred_idx][0]
    heatmap = np.sum(np.abs(shap_img), axis=-1)
    return heatmap / (heatmap.max() + 1e-10)

# EVALUATION METRICS
def evaluate_explanation(model, img, pred_idx, heatmap, predict_fn=None):
    """Evaluate explanation quality with AOPC and faithfulness metrics"""
    flat = heatmap.flatten()
    H, W = heatmap.shape

    # Deletion & Insertion curves
    del_curve, ins_curve = [], []
    for i in range(0, 101, 5):
        ratio = i / 100
        pixels = int(ratio * len(flat))

        # Deletion
        order_del = np.argsort(flat)[::-1][:pixels]
        test_img_del = img[0].copy()
        if pixels > 0:
            rows, cols = np.unravel_index(order_del, (H, W))
            test_img_del[rows, cols] = 0

        # Insertion
        order_ins = np.argsort(flat)[-pixels:] if pixels > 0 else []
        test_img_ins = np.zeros_like(img[0])
        if pixels > 0:
            rows, cols = np.unravel_index(order_ins, (H, W))
            test_img_ins[rows, cols] = img[0][rows, cols]

        # Predict
        if predict_fn:
            del_curve.append(predict_fn(test_img_del[None, ...])[0][pred_idx])
            ins_curve.append(predict_fn(test_img_ins[None, ...])[0][pred_idx])
        else:
            del_curve.append(model.predict(test_img_del[None, ...], verbose=0)[0][pred_idx])
            ins_curve.append(model.predict(test_img_ins[None, ...], verbose=0)[0][pred_idx])

    # Faithfulness (Spearman correlation)
    correlations = []
    for _ in range(50):
        x1, y1 = np.random.randint(0, W-20), np.random.randint(0, H-20)
        region_attr = np.mean(heatmap[y1:y1+20, x1:x1+20])

        masked = img[0].copy()
        masked[y1:y1+20, x1:x1+20] = 0

        if predict_fn:
            orig_conf = predict_fn(img)[0][pred_idx]
            masked_conf = predict_fn(masked[None, ...])[0][pred_idx]
        else:
            orig_conf = model.predict(img, verbose=0)[0][pred_idx]
            masked_conf = model.predict(masked[None, ...], verbose=0)[0][pred_idx]

        correlations.append((region_attr, orig_conf - masked_conf))

    spearman, _ = spearmanr([c[0] for c in correlations], [c[1] for c in correlations])

    return {
        'aopc_del': np.trapz(del_curve, dx=0.05),
        'aopc_ins': np.trapz(ins_curve, dx=0.05),
        'spearman': spearman,
        'del_curve': del_curve,
        'ins_curve': ins_curve
    }

# MAIN EVALUATION
def run_evaluation(test_gen, train_gen, class_names):
    """Run comprehensive XAI evaluation"""
    print("="*80)
    print("XAI EVALUATION FRAMEWORK")
    print("="*80)

    # Load models
    models = load_models()
    print(f"\nActive Models: {[k for k, v in models.items() if v and k != 'prep']}")

    # Get test data
    test_gen.reset()
    X_test, y_test = next(test_gen)

    # Get background for SHAP
    train_gen.reset()
    background = next(train_gen)[0][:BACKGROUND_SAMPLES]

    results = []
    all_heatmaps = {}
    all_curves = {}

    # Evaluate each image
    for img_idx in EVAL_IMAGES:
        print(f"\n{'='*80}")
        print(f"Evaluating Image #{img_idx}")
        print('='*80)
        img = X_test[img_idx][None, ...]
        true_label = class_names[np.argmax(y_test[img_idx])]

        # Evaluate all CNN models including CNN-XGBoost
        for model_name in ['ResNet50', 'DenseNet121', 'CNN-XGBoost']:
            model = models[model_name]
            if not model:
                continue

            pred_idx = np.argmax(model.predict(img, verbose=0)[0])
            pred_label = class_names[pred_idx]

            print(f"\n{model_name}: True={true_label}, Pred={pred_label}")

            methods = {
                'GradCAM': lambda: gradcam(img, model, CONV_LAYERS[model_name], pred_idx),
                'IntGrad': lambda: integrated_gradients(img, model, pred_idx),
                'LIME': lambda: lime_explain(img, model, pred_idx),
                'SHAP': lambda: shap_explain(img, model, background, pred_idx)
            }

            for method_name, method_fn in methods.items():
                try:
                    heatmap = gaussian_filter(method_fn(), sigma=2)
                    metrics = evaluate_explanation(model, img, pred_idx, heatmap)

                    results.append({
                        'Image': img_idx,
                        'Model': model_name,
                        'Method': method_name,
                        'True': true_label,
                        'Pred': pred_label,
                        'AOPC_Del': metrics['aopc_del'],
                        'AOPC_Ins': metrics['aopc_ins'],
                        'Spearman': metrics['spearman']
                    })

                    all_heatmaps[(img_idx, model_name, method_name)] = heatmap
                    all_curves[(img_idx, model_name, method_name)] = {
                        'del': metrics['del_curve'],
                        'ins': metrics['ins_curve']
                    }

                    print(f"  ✓ {method_name}: AOPC_Del={metrics['aopc_del']:.3f}, "
                          f"AOPC_Ins={metrics['aopc_ins']:.3f}, Spearman={metrics['spearman']:.3f}")
                except Exception as e:
                    print(f"  ✗ {method_name}: {str(e)[:50]}")

    return pd.DataFrame(results), all_heatmaps, all_curves

# ENHANCED VISUALIZATIONS
def visualize_results(df, heatmaps, curves, X_test):
    """Generate comprehensive visualizations"""
    print("\n" + "="*80)
    print("GENERATING VISUALIZATIONS")
    print("="*80)

    # 1. Performance Heatmap
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    metrics = ['AOPC_Del', 'AOPC_Ins', 'Spearman']
    titles = ['AOPC Deletion\n(Lower = Better)', 'AOPC Insertion\n(Higher = Better)',
              'Faithfulness (Spearman)\n(Higher = Better)']

    for idx, (metric, title) in enumerate(zip(metrics, titles)):
        pivot = df.groupby(['Model', 'Method'])[metric].mean().unstack()
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn' if idx != 0 else 'RdYlGn_r',
                   linewidths=2, ax=axes[idx], cbar_kws={'shrink': 0.8}, vmin=0, vmax=1)
        axes[idx].set_title(title, fontweight='bold', fontsize=12)
        axes[idx].set_xlabel('')
        axes[idx].set_ylabel('Model' if idx == 0 else '', fontweight='bold')
        axes[idx].tick_params(labelsize=10)

    plt.tight_layout()
    plt.savefig('xai_performance_metrics.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: xai_performance_metrics.png")
    plt.show()
    plt.close()

    # 2. Rankings with Model Comparison
    # Composite score computed directly from the three metrics
    # (no cross-method normalization), so it is reproducible from
    # its own definition:
    #   Score = (1 - AOPC_Del) * 0.25 + AOPC_Ins * 0.25 + Spearman * 0.50
    ranking = df.groupby(['Model', 'Method']).mean(numeric_only=True)
    ranking['Score'] = (
        (1 - ranking['AOPC_Del']) * 0.25 +
        ranking['AOPC_Ins'] * 0.25 +
        ranking['Spearman'] * 0.50
    )
    ranking = ranking.sort_values('Score', ascending=False)

    fig, ax = plt.subplots(figsize=(12, 8))
    colors = {'ResNet50': '#FF6B6B', 'DenseNet121': '#4ECDC4', 'CNN-XGBoost': '#95E1D3'}
    bar_colors = [colors.get(idx[0], 'gray') for idx in ranking.index]

    bars = ax.barh(range(len(ranking)), ranking['Score'], color=bar_colors, edgecolor='black', linewidth=1.5)
    ax.set_yticks(range(len(ranking)))
    ax.set_yticklabels([f"{idx[0]}\n{idx[1]}" for idx in ranking.index], fontsize=10, fontweight='bold')
    ax.set_xlabel('Overall Score (0-1)', fontweight='bold', fontsize=12)
    ax.set_title('XAI Method Rankings Across All Models', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.set_xlim(0, 1.1)

    for i, score in enumerate(ranking['Score']):
        ax.text(score + 0.02, i, f'{score:.3f}', va='center', fontweight='bold', fontsize=9)

    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=colors[k], label=k) for k in colors.keys()]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

    plt.tight_layout()
    plt.savefig('xai_rankings.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: xai_rankings.png")
    plt.close()

    # 3. Deletion/Insertion Curves
    img_idx = EVAL_IMAGES[0]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    x_vals = np.arange(0, 101, 5) / 100

    models_to_plot = ['ResNet50', 'DenseNet121', 'CNN-XGBoost']
    methods_to_plot = ['GradCAM', 'IntGrad', 'LIME', 'SHAP']

    for idx, method in enumerate(methods_to_plot):
        row, col = idx // 2, idx % 2
        ax = axes[row, col]

        for model in models_to_plot:
            key = (img_idx, model, method)
            if key in curves:
                color = colors.get(model, 'gray')
                ax.plot(x_vals, curves[key]['del'], label=f'{model} (Del)',
                       color=color, linestyle='--', linewidth=2, marker='o', markersize=3)
                ax.plot(x_vals, curves[key]['ins'], label=f'{model} (Ins)',
                       color=color, linestyle='-', linewidth=2, marker='s', markersize=3)

        ax.set_xlabel('Fraction of Pixels Modified', fontweight='bold')
        ax.set_ylabel('Prediction Confidence', fontweight='bold')
        ax.set_title(f'{method} - Deletion & Insertion', fontweight='bold', fontsize=11)
        ax.legend(fontsize=8, ncol=2)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('xai_curves.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: xai_curves.png")
    plt.close()

    # 4. Heatmap Comparison Grid (All 3 Models)
    img_idx = EVAL_IMAGES[0]
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    methods = ['GradCAM', 'IntGrad', 'LIME', 'SHAP']

    for i, model in enumerate(['ResNet50', 'DenseNet121', 'CNN-XGBoost']):
        for j, method in enumerate(methods):
            key = (img_idx, model, method)
            if key in heatmaps:
                im = axes[i, j].imshow(heatmaps[key], cmap='jet', interpolation='bilinear')
                axes[i, j].set_title(f'{model}\n{method}', fontweight='bold', fontsize=10)
                plt.colorbar(im, ax=axes[i, j], fraction=0.046, pad=0.04)
            else:
                axes[i, j].text(0.5, 0.5, 'N/A', ha='center', va='center',
                              fontsize=14, fontweight='bold', color='red')
                axes[i, j].set_title(f'{model}\n{method}', fontweight='bold', fontsize=10)
            axes[i, j].axis('off')

    plt.suptitle(f'XAI Heatmap Comparison - Image #{img_idx}',
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig('xai_heatmap_comparison.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: xai_heatmap_comparison.png")
    plt.close()

    # 5. Method Comparison by Model
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    models_list = df['Model'].unique()

    for idx, model in enumerate(models_list):
        model_df = df[df['Model'] == model]
        method_scores = model_df.groupby('Method').mean(numeric_only=True)

        # Same composite score definition as above, applied per model.
        method_scores['Score'] = (
            (1 - method_scores['AOPC_Del']) * 0.25 +
            method_scores['AOPC_Ins'] * 0.25 +
            method_scores['Spearman'] * 0.50
        )

        method_scores = method_scores.sort_values('Score', ascending=False)

        bars = axes[idx].bar(range(len(method_scores)), method_scores['Score'],
                            color=colors.get(model, 'gray'), edgecolor='black', linewidth=1.5)
        axes[idx].set_xticks(range(len(method_scores)))
        axes[idx].set_xticklabels(method_scores.index, fontweight='bold', rotation=45)
        axes[idx].set_ylabel('Score', fontweight='bold')
        axes[idx].set_title(f'{model}', fontweight='bold', fontsize=12)
        axes[idx].grid(axis='y', alpha=0.3)
        axes[idx].set_ylim(0, 1.1)

        for i, score in enumerate(method_scores['Score']):
            axes[idx].text(i, score + 0.02, f'{score:.3f}', ha='center',
                          fontweight='bold', fontsize=9)

    plt.tight_layout()
    plt.savefig('xai_method_by_model.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: xai_method_by_model.png")
    plt.close()

    return ranking

def generate_report(df, ranking):
    """Generate summary report"""
    best = ranking.iloc[0]

    report = f"""
{'='*80}
XAI EVALUATION REPORT
{'='*80}

BEST PERFORMING COMBINATION:
  Model:          {best.name[0]}
  Method:         {best.name[1]}
  Overall Score:  {best['Score']:.4f}
  Faithfulness:   {best['Spearman']:.4f}
  AOPC Deletion:  {best['AOPC_Del']:.4f}
  AOPC Insertion: {best['AOPC_Ins']:.4f}

TOP 5 METHODS:
"""
    for i, (idx, row) in enumerate(ranking.head(5).iterrows(), 1):
        report += f"  {i}. {idx[0]:15s} - {idx[1]:10s} (Score: {row['Score']:.4f})\n"

    report += f"\n{'='*80}\nAVERAGE PERFORMANCE BY MODEL-METHOD:\n{'='*80}\n"
    numeric_cols = ['AOPC_Del', 'AOPC_Ins', 'Spearman']
    avg_performance = df.groupby(['Model', 'Method'])[numeric_cols].mean().round(4)
    report += str(avg_performance) + "\n"

    report += f"\n{'='*80}\nBEST METHOD PER MODEL:\n{'='*80}\n"
    for model in df['Model'].unique():
        model_ranking = ranking[ranking.index.get_level_values(0) == model]
        if len(model_ranking) > 0:
            best_method = model_ranking.iloc[0]
            report += f"  {model:15s}: {best_method.name[1]:10s} (Score: {best_method['Score']:.4f})\n"

    with open('xai_report.txt', 'w') as f:
        f.write(report)

    df.to_csv('xai_detailed_results.csv', index=False)
    ranking.to_csv('xai_rankings.csv')

    print("\n✓ Saved: xai_report.txt")
    print("✓ Saved: xai_detailed_results.csv")
    print("✓ Saved: xai_rankings.csv")

    return report

# EXECUTION
if __name__ == "__main__":
    # Run evaluation
    df, heatmaps, curves = run_evaluation(test_generator, train_generator, class_names)

    # Generate visualizations
    ranking = visualize_results(df, heatmaps, curves, X_test_batch)

    # Generate report
    report = generate_report(df, ranking)
    print(report)

    print("\n" + "="*80)
    print("EVALUATION COMPLETE!")
    print("="*80)

In [ ]:
# ============================================================
# XAI COMPOSITE SCORE — FINAL SUMMARY
# Aggregates AOPC Deletion, AOPC Insertion, and Faithfulness
# (Spearman correlation) into a single composite interpretability
# score per XAI method, then produces the comparison table and
# ranking plot used for reporting.
#
# Composite score definition:
#   Score = (1 - AOPC_Deletion) * 0.25
#         +  AOPC_Insertion     * 0.25
#         +  Faithfulness       * 0.50
#
# AOPC Deletion is inverted (1 - value) because lower deletion
# scores indicate stronger localization (removing fewer
# top-attributed pixels causes a larger prediction drop). This
# formula is applied directly to each method's raw metrics, with
# no cross-method normalization, so the Score column is always
# reproducible straight from the AOPC/Faithfulness values it is
# built from.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
XAI_COLORS = {'GradCAM': '#E74C3C', 'IntGrad': '#3498DB', 'LIME': '#2ECC71', 'SHAP': '#F39C12'}


def compute_composite_score(aopc_del, aopc_ins, faithfulness):
    """Composite interpretability score, directly reproducible
    from its three input metrics (no cross-method normalization)."""
    return (1 - aopc_del) * 0.25 + aopc_ins * 0.25 + faithfulness * 0.50


print("\n" + "=" * 80)
print("XAI TECHNIQUE COMPARISON — FINAL SUMMARY")
print("=" * 80 + "\n")

# Load per-image / per-model / per-method results from the full
# evaluation run if available, and average to the method level.
try:
    df = pd.read_csv('xai_detailed_results.csv')
    method_avg = df.groupby('Method')[['AOPC_Del', 'AOPC_Ins', 'Spearman']].mean()
    method_avg.columns = ['AOPC_Deletion', 'AOPC_Insertion', 'Faithfulness']
    print(f"Loaded {len(df)} evaluations from xai_detailed_results.csv")
except FileNotFoundError:
    raise FileNotFoundError(
        "xai_detailed_results.csv not found. Run the evaluation cell above first "
        "(run_evaluation -> visualize_results -> generate_report)."
    )

method_avg['Score'] = compute_composite_score(
    method_avg['AOPC_Deletion'], method_avg['AOPC_Insertion'], method_avg['Faithfulness']
)
method_avg = method_avg.sort_values('Score', ascending=False)

print("\nXAI Technique Performance Summary")
print("-" * 60)
for method, row in method_avg.iterrows():
    print(f"{method:10s} -> Score: {row['Score']:.4f} | "
          f"Del: {row['AOPC_Deletion']:.4f} | Ins: {row['AOPC_Insertion']:.4f} | "
          f"Faith: {row['Faithfulness']:.4f}")

method_avg.to_csv('xai_method_comparison.csv')
print("\nSaved: xai_method_comparison.csv")

# ---- Rankings bar chart ----
fig, ax = plt.subplots(figsize=(10, 5))
methods = method_avg.index.tolist()
scores = method_avg['Score'].values
colors = [XAI_COLORS.get(m, '#95A5A6') for m in methods]

bars = ax.barh(methods, scores, color=colors, alpha=0.85, edgecolor='black', linewidth=2)
ax.set_xlabel('Composite Score', fontweight='bold')
ax.set_title('XAI Technique Rankings (Composite Interpretability Score)',
             fontweight='bold', fontsize=13)
ax.set_xlim(0, 1.0)
for bar, score in zip(bars, scores):
    ax.text(score + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{score:.3f}', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('xai_rankings_final.png', dpi=300, bbox_inches='tight')
plt.show()

# ---- Metric breakdown bar chart ----
fig, ax = plt.subplots(figsize=(12, 5))
metric_cols = ['AOPC_Deletion', 'AOPC_Insertion', 'Faithfulness', 'Score']
x = np.arange(len(metric_cols))
width = 0.2

for i, method in enumerate(methods):
    values = [method_avg.loc[method, m] for m in metric_cols]
    ax.bar(x + i * width, values, width, label=method,
           color=XAI_COLORS.get(method, '#95A5A6'), alpha=0.85,
           edgecolor='black', linewidth=1.5)

ax.set_xlabel('Metric', fontweight='bold')
ax.set_ylabel('Value', fontweight='bold')
ax.set_title('XAI Performance Comparison', fontweight='bold', fontsize=13)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(['AOPC Del \u2193', 'AOPC Ins \u2191', 'Faithfulness \u2191', 'Composite Score \u2191'])
ax.legend()
ax.set_ylim([0, 1.0])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('xai_overall_metrics_final.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "=" * 80)
print(f"BEST XAI TECHNIQUE BY COMPOSITE SCORE: {method_avg.index[0]} "
      f"(Score: {method_avg.iloc[0]['Score']:.4f})")
print("=" * 80)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
import os

# Define paths (use your existing paths)
train_dir = os.path.join("BrainTumour", "split_dataset", "train")
IMG_SIZE = (224, 224)

# Create the same data augmentation setup as in your training code
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.3,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Function to visualize original vs augmented images
def visualize_augmentation(image_path, num_augmented=5):
    """
    Display original image alongside multiple augmented versions

    Args:
        image_path: Path to the image file
        num_augmented: Number of augmented versions to show
    """
    # Load and preprocess the image
    img = load_img(image_path, target_size=IMG_SIZE)
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    # Create figure
    fig, axes = plt.subplots(1, num_augmented + 1, figsize=(20, 4))

    # Show original image
    axes[0].imshow(img)
    axes[0].set_title("Original Image", fontsize=12, fontweight='bold')
    axes[0].axis('off')

    # Generate and show augmented images
    aug_iter = train_datagen.flow(img_array, batch_size=1)

    for i in range(num_augmented):
        aug_img = next(aug_iter)[0]
        axes[i + 1].imshow(aug_img)
        axes[i + 1].set_title(f"Augmented {i+1}", fontsize=12)
        axes[i + 1].axis('off')

    plt.tight_layout()
    plt.show()


# Function to show multiple samples from each class
def visualize_class_samples(train_dir, samples_per_class=3, augmentations_per_sample=3):
    """
    Display original and augmented images for each class

    Args:
        train_dir: Directory containing training images
        samples_per_class: Number of original images to show per class
        augmentations_per_sample: Number of augmented versions per image
    """
    class_names = sorted(os.listdir(train_dir))

    for class_name in class_names:
        class_path = os.path.join(train_dir, class_name)
        if not os.path.isdir(class_path):
            continue

        # Get image files
        image_files = [f for f in os.listdir(class_path)
                      if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]

        # Select random samples
        selected_images = np.random.choice(image_files,
                                          min(samples_per_class, len(image_files)),
                                          replace=False)

        print(f"\n{'='*80}")
        print(f"CLASS: {class_name}")
        print(f"{'='*80}")

        for img_file in selected_images:
            img_path = os.path.join(class_path, img_file)
            print(f"\nShowing augmentations for: {img_file}")
            visualize_augmentation(img_path, num_augmented=augmentations_per_sample)


# Function to compare original vs preprocessed (rescaled only)
def compare_preprocessing(image_path):
    """
    Compare original image with preprocessed (rescaled) version
    """
    # Load original
    img_original = load_img(image_path, target_size=IMG_SIZE)

    # Load and preprocess
    img_array = img_to_array(img_original)
    img_array = np.expand_dims(img_array, axis=0)

    # Apply only rescaling (preprocessing without augmentation)
    preprocess_only = ImageDataGenerator(rescale=1./255)
    preprocessed = preprocess_only.flow(img_array, batch_size=1)
    img_preprocessed = next(preprocessed)[0]

    # Display
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].imshow(img_original)
    axes[0].set_title("Original Image\n(0-255 range)", fontsize=12, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(img_preprocessed)
    axes[1].set_title("Preprocessed Image\n(0-1 range, rescaled)", fontsize=12, fontweight='bold')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

    print(f"Original pixel range: [{np.array(img_original).min()}, {np.array(img_original).max()}]")
    print(f"Preprocessed pixel range: [{img_preprocessed.min():.4f}, {img_preprocessed.max():.4f}]")


# Function to visualize augmentation effects individually
def visualize_individual_augmentations(image_path):
    """
    Show the effect of each augmentation technique separately
    """
    img = load_img(image_path, target_size=IMG_SIZE)
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    # Define individual augmentations
    augmentations = {
        'Original': ImageDataGenerator(rescale=1./255),
        'Rotation': ImageDataGenerator(rescale=1./255, rotation_range=40),
        'Width Shift': ImageDataGenerator(rescale=1./255, width_shift_range=0.3),
        'Height Shift': ImageDataGenerator(rescale=1./255, height_shift_range=0.3),
        'Shear': ImageDataGenerator(rescale=1./255, shear_range=0.3),
        'Zoom': ImageDataGenerator(rescale=1./255, zoom_range=0.3),
        'Horizontal Flip': ImageDataGenerator(rescale=1./255, horizontal_flip=True),
        'Brightness': ImageDataGenerator(rescale=1./255, brightness_range=[0.8, 1.2])
    }

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.ravel()

    for idx, (name, datagen) in enumerate(augmentations.items()):
        aug_iter = datagen.flow(img_array, batch_size=1, seed=42)
        aug_img = next(aug_iter)[0]

        axes[idx].imshow(aug_img)
        axes[idx].set_title(name, fontsize=11, fontweight='bold')
        axes[idx].axis('off')

    plt.suptitle("Individual Augmentation Effects", fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


# MAIN EXECUTION

print("Brain Tumor MRI Dataset - Image Augmentation Visualization")
print("="*80)

# Example 1: Show samples from each class with augmentations
print("\n1. Visualizing samples from each class with augmentations...")
visualize_class_samples(train_dir, samples_per_class=2, augmentations_per_sample=4)

# Example 2: Compare original vs preprocessed
print("\n2. Comparing original vs preprocessed image...")
# Get a sample image
sample_class = os.listdir(train_dir)[0]
sample_image_path = os.path.join(train_dir, sample_class,
                                 os.listdir(os.path.join(train_dir, sample_class))[0])
compare_preprocessing(sample_image_path)

# Example 3: Show individual augmentation effects
print("\n3. Visualizing individual augmentation effects...")
visualize_individual_augmentations(sample_image_path)

print("\n" + "="*80)
print("Visualization complete!")
print("="*80)